# 📈 Proyeksi SARIMA / SARIMAX — Google Colab

Notebook ini menjalankan model **SARIMA** dan **SARIMAX** (SARIMA dengan variabel independen X) untuk proyeksi
deret waktu tanpa perlu menginstal apa pun, untuk **data apa saja**: harian, mingguan, bulanan, triwulanan, atau tahunan.
Mesin perhitungannya **sama** dengan *ARIMA Forecasting Platform* (aplikasi web).

**Cara memakai:** jalankan sel **satu per satu dari atas ke bawah** dengan tombol ▶ di kiri setiap sel
(atau Shift+Enter). Isi pengaturan di kotak formulir sebelah kanan sel, lalu jalankan ulang sel tersebut.
Kolom, frekuensi, periode latih/uji, panjang musim, dan horizon **dipilih otomatis dari data** bila dikosongkan.

**SARIMAX:** semua kolom angka selain tanggal dan target otomatis menjadi variabel X (bisa diubah di Langkah 3).
Model dengan X selalu dibandingkan dengan model yang sama **tanpa X** pada sampel yang sama. Nilai X masa depan
bisa **diisi manual** (baris tanggal ke depan di file: X terisi, target kosong) dan/atau **diproyeksikan dengan SARIMA**:
- **ex-post**: ramalan memakai X yang diketahui (X aktual di periode uji / X diisi manual ke depan);
- **ex-ante**: ramalan memakai X hasil proyeksi SARIMA (kondisi nyata saat X belum diketahui).

| Langkah | Isi |
|---|---|
| 1 | Menyiapkan mesin |
| 2 | Mengunggah data |
| 3 | Memilih kolom (tanggal, target, X) & memeriksa data |
| 4 | Pengaturan model (termasuk pemilihan orde otomatis) |
| 5 | Menjalankan model & membandingkan dengan/tanpa X |
| 6 | Diagnostik residual |
| 7 | Grafik & tabel hasil |
| 8 | Evaluasi periode uji |
| 9 | Mengunduh hasil |
| + | Uji otomatis & tampilan web *(opsional)* |

**Metode estimasi:** *exact maximum likelihood* (semua observasi ikut dihitung, tanpa burn-in),
kovarians koefisien OPG, uji-t dengan derajat bebas n − k, differencing dilakukan eksplisit pada Y dan X sebelum
estimasi, dan konstanta **C** diestimasi sebagai rata-rata deret (atau drift bila ada differencing).
Kriteria informasi dilaporkan per observasi.

> 🔒 Data yang diunggah hanya ada di sesi Colab Anda dan hilang saat sesi ditutup.


In [ ]:
#@title Langkah 1 — Siapkan mesin SARIMA (cukup dijalankan, tidak perlu diubah)
import os, sys, importlib
from pathlib import Path

FOLDER = Path("/content/sarima_app")
FOLDER.mkdir(parents=True, exist_ok=True)
os.chdir(FOLDER)
if str(FOLDER) not in sys.path:
    sys.path.insert(0, str(FOLDER))

BERKAS_MESIN = {
    'modules/__init__.py': '',
    'modules/data_loader.py': '"""Membaca file CSV/Excel dan meringkas isinya. Tidak mengubah data."""\nfrom __future__ import annotations\n\nimport pandas as pd\n\n\nclass GalatData(Exception):\n    """Galat yang pesannya aman ditampilkan langsung ke pengguna."""\n\n\ndef baca_file(file) -> pd.DataFrame:\n    """Baca CSV atau Excel. `file` boleh path atau objek unggahan Streamlit."""\n    nama = getattr(file, "name", str(file)).lower()\n    try:\n        if nama.endswith((".xlsx", ".xls")):\n            df = pd.read_excel(file)\n        elif nama.endswith(".csv"):\n            df = pd.read_csv(file, sep=None, engine="python")  # deteksi koma / titik koma\n        else:\n            raise GalatData("Format file tidak didukung. Gunakan .csv atau .xlsx.")\n    except GalatData:\n        raise\n    except Exception as e:\n        raise GalatData(f"File tidak bisa dibaca ({e}). Pastikan file tidak rusak dan tidak sedang dikunci.") from e\n\n    if df.empty or df.shape[1] < 2:\n        raise GalatData("File harus berisi minimal dua kolom (tanggal dan nilai) dan tidak boleh kosong.")\n    df.columns = [str(c).strip() for c in df.columns]\n    return df\n\n\ndef ringkasan_dataset(df: pd.DataFrame) -> dict:\n    """Ringkasan untuk halaman unggah: jumlah baris/kolom, tipe data, nilai kosong."""\n    info = pd.DataFrame({\n        "Tipe data": df.dtypes.astype(str),\n        "Nilai kosong": df.isna().sum(),\n        "Contoh nilai": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],\n    })\n    info["Contoh nilai"] = info["Contoh nilai"].astype(str)\n    return {"jumlah_baris": len(df), "jumlah_kolom": df.shape[1], "info_kolom": info}\n\n\ndef tebak_kolom(df: pd.DataFrame) -> tuple[str, str]:\n    """Tebakan awal kolom tanggal dan kolom target (pengguna tetap bisa mengganti)."""\n    kolom = list(df.columns)\n    kata_tanggal = ("tanggal", "periode", "date", "bulan", "waktu", "time", "tahun")\n    tgl = next((c for c in kolom if any(k in c.lower() for k in kata_tanggal)), None)\n    if tgl is None:\n        tgl = next((c for c in kolom if pd.api.types.is_datetime64_any_dtype(df[c])), kolom[0])\n    angka = [c for c in kolom if c != tgl and pd.api.types.is_numeric_dtype(df[c])]\n    target = angka[0] if angka else next(c for c in kolom if c != tgl)\n    return tgl, target\n',
    'modules/data_preprocessing.py': '"""\nMembentuk deret waktu dari tabel, memeriksa masalah data, dan transformasi log.\n\nAturan: data tidak pernah dihapus atau diubah diam-diam. Setiap periode yang\nhilang, nilai kosong, atau nilai yang tidak bisa dibaca dilaporkan ke pengguna.\n"""\nfrom __future__ import annotations\n\nimport re\nimport warnings\nfrom dataclasses import dataclass, field\n\nimport numpy as np\nimport pandas as pd\n\n# nama tampilan -> (kode periode pandas, panjang musim bawaan)\nFREKUENSI = {\n    "Bulanan": ("M", 12),\n    "Triwulanan": ("Q", 4),\n    "Tahunan": ("Y", 1),\n    "Mingguan": ("W", 52),\n    "Harian": ("D", 7),\n}\n\nTRANSFORMASI = {\n    "log": "Terapkan log natural (data diunggah dalam nilai asli)",\n    "sudah_log": "Data sudah dalam bentuk log (jangan di-log lagi)",\n    "tanpa": "Tanpa transformasi",\n}\n\n_POLA_BULANAN = re.compile(r"^\\s*(\\d{4})\\s*[Mm]\\s*(\\d{1,2})\\s*$")   # 2021M01\n_POLA_TRIWULAN = re.compile(r"^\\s*(\\d{4})\\s*[Qq]\\s*([1-4])\\s*$")   # 2021Q1\n_POLA_TAHUN = re.compile(r"^\\s*(\\d{4})(?:\\.0+)?\\s*$")                  # 2021\n\n\ndef ke_periode(kolom: pd.Series, kode_frek: str) -> pd.PeriodIndex:\n    """Ubah kolom tanggal ke PeriodIndex. Juga mengenali format 2021M01 dan 2021Q1."""\n    teks = kolom.astype(str).str.strip()\n    if kode_frek == "M" and teks.str.match(_POLA_BULANAN).all():\n        b = teks.str.extract(_POLA_BULANAN)\n        return pd.PeriodIndex(b[0] + "-" + b[1].str.zfill(2), freq="M")\n    if kode_frek == "Q" and teks.str.match(_POLA_TRIWULAN).all():\n        b = teks.str.extract(_POLA_TRIWULAN)\n        return pd.PeriodIndex(b[0] + "Q" + b[1], freq="Q")\n    if teks.str.match(_POLA_TAHUN).all():\n        tahun = teks.str.extract(_POLA_TAHUN)[0]\n        if kode_frek == "Y":\n            return pd.PeriodIndex(tahun, freq="Y")\n        raise ValueError("Kolom tanggal hanya berisi tahun; pilih frekuensi Tahunan.")\n    if pd.api.types.is_datetime64_any_dtype(kolom):\n        tanggal = pd.DatetimeIndex(kolom)\n    else:\n        # Teks tanggal: format Indonesia hari/bulan/tahun (01/02/2021 = 1 Februari). ISO (2021-02-01) tetap benar.\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore", UserWarning)\n            tanggal = pd.DatetimeIndex(pd.to_datetime(kolom, errors="raise", dayfirst=True))\n    return tanggal.to_period(kode_frek)\n\n\ndef tebak_frekuensi(kolom: pd.Series) -> str:\n    """Tebak frekuensi data dari kolom tanggal (jarak median antar-tanggal). Bawaan: Bulanan."""\n    teks = kolom.dropna().astype(str).str.strip()\n    if len(teks) and teks.str.match(_POLA_BULANAN).all():\n        return "Bulanan"\n    if len(teks) and teks.str.match(_POLA_TRIWULAN).all():\n        return "Triwulanan"\n    if len(teks) and teks.str.match(_POLA_TAHUN).all():\n        return "Tahunan"\n    try:\n        if pd.api.types.is_datetime64_any_dtype(kolom):\n            tanggal = pd.DatetimeIndex(kolom.dropna())\n        else:\n            with warnings.catch_warnings():\n                warnings.simplefilter("ignore", UserWarning)\n                tanggal = pd.DatetimeIndex(pd.to_datetime(kolom.dropna(), errors="raise", dayfirst=True))\n    except Exception:\n        return "Bulanan"\n    jarak = pd.Series(tanggal.sort_values()).diff().dt.days.dropna()\n    if jarak.empty:\n        return "Bulanan"\n    hari = float(jarak.median())\n    if hari <= 1.5:\n        return "Harian"\n    if hari <= 8:\n        return "Mingguan"\n    if hari <= 45:\n        return "Bulanan"\n    if hari <= 135:\n        return "Triwulanan"\n    return "Tahunan"\n\n\ndef label_periode(p: pd.Period) -> str:\n    """Label ringkas untuk tampilan: 2021-01, 2021Q1, 2021, atau tanggal awal minggu/hari."""\n    kode = p.freqstr[0]\n    if kode in ("Y", "A"):\n        return str(p.year)\n    if kode == "M":\n        return p.strftime("%Y-%m")\n    if kode == "Q":\n        return f"{p.year}Q{p.quarter}"\n    return p.start_time.strftime("%Y-%m-%d")\n\n\ndef ke_angka(kolom: pd.Series) -> pd.Series:\n    """Ubah teks ke angka. Mengenali format Indonesia (1.234.567,89)."""\n    if pd.api.types.is_numeric_dtype(kolom):\n        return kolom.astype(float)\n    teks = kolom.astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})\n    format_id = teks.str.contains(r",\\d+$", na=False) | teks.str.contains(r"\\.\\d{3}(?:\\.|$)", na=False)\n    teks = teks.where(~format_id, teks.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))\n    return pd.to_numeric(teks, errors="coerce")\n\n\n@dataclass\nclass HasilValidasi:\n    seri: pd.Series\n    galat: list[str] = field(default_factory=list)\n    peringatan: list[str] = field(default_factory=list)\n\n    @property\n    def ok(self) -> bool:\n        return not self.galat\n\n\ndef siapkan_seri(df: pd.DataFrame, kolom_tanggal: str, kolom_target: str,\n                 frekuensi: str = "Bulanan") -> HasilValidasi:\n    """Bentuk deret waktu reguler dan laporkan semua masalah data."""\n    kode, _ = FREKUENSI[frekuensi]\n    galat, peringatan = [], []\n    if kolom_tanggal == kolom_target:\n        return HasilValidasi(pd.Series(dtype=float), ["Kolom tanggal dan kolom target tidak boleh sama."])\n\n    tanggal_kosong = df[kolom_tanggal].isna()\n    if tanggal_kosong.any():\n        peringatan.append(f"{int(tanggal_kosong.sum())} baris tanpa tanggal tidak diikutkan.")\n    df = df.loc[~tanggal_kosong]\n\n    try:\n        periode = ke_periode(df[kolom_tanggal], kode)\n    except Exception:\n        contoh = ", ".join(df[kolom_tanggal].astype(str).head(3))\n        return HasilValidasi(pd.Series(dtype=float), [\n            f"Kolom tanggal \'{kolom_tanggal}\' tidak bisa dibaca sebagai tanggal (contoh isi: {contoh}). "\n            "Gunakan format seperti 2021-01-31, 31/01/2021, 2021M01, 2021Q1, atau 2021 (tahunan)."])\n\n    asli = df[kolom_target]\n    nilai = ke_angka(asli)\n    tak_terbaca = nilai.isna() & asli.notna()\n    if len(nilai) and tak_terbaca.mean() > 0.5:\n        return HasilValidasi(pd.Series(dtype=float), [\n            f"Kolom \'{kolom_target}\' sebagian besar bukan angka. Pilih kolom target yang berisi nilai numerik."])\n    if tak_terbaca.any():\n        peringatan.append(f"{int(tak_terbaca.sum())} nilai di kolom target bukan angka dan dianggap kosong.")\n\n    seri = pd.Series(nilai.to_numpy(), index=periode, name=kolom_target).sort_index()\n\n    if seri.index.duplicated().any():\n        dup = sorted({label_periode(p) for p in seri.index[seri.index.duplicated()]})\n        galat.append(\n            f"Ada {len(dup)} periode ganda untuk frekuensi {frekuensi.lower()} (misalnya {\', \'.join(dup[:5])}). "\n            "Periksa apakah frekuensi yang dipilih sudah benar, atau agregasikan data terlebih dahulu.")\n        return HasilValidasi(seri, galat, peringatan)\n\n    lengkap = pd.period_range(seri.index.min(), seri.index.max(), freq=seri.index.freq)\n    hilang = lengkap.difference(seri.index)\n    if len(hilang):\n        peringatan.append(\n            f"{len(hilang)} periode tidak ada di data ({\', \'.join(label_periode(p) for p in hilang[:6])}"\n            f"{\' ...\' if len(hilang) > 6 else \'\'}). Periode ini dibiarkan kosong; model tidak mengisinya dengan nilai karangan.")\n    seri = seri.reindex(lengkap)\n\n    terakhir = seri.last_valid_index()\n    masa_depan = seri.index > terakhir if terakhir is not None else np.zeros(len(seri), bool)\n    if masa_depan.any():\n        peringatan.append(f"{int(masa_depan.sum())} periode di akhir data tidak punya nilai target "\n                          f"({label_periode(seri.index[masa_depan][0])} s.d. {label_periode(seri.index[-1])}); "\n                          "dibaca sebagai periode masa depan (mis. berisi nilai X untuk proyeksi).")\n    kosong = seri.isna() & ~seri.index.isin(hilang) & ~masa_depan\n    if kosong.any() and not tak_terbaca.any():\n        peringatan.append(f"{int(kosong.sum())} nilai target kosong.")\n\n    if seri.notna().sum() < 10:\n        galat.append("Data terlalu sedikit (kurang dari 10 observasi terisi) untuk model ARIMA.")\n    return HasilValidasi(seri, galat, peringatan)\n\n\n# --------------------------------------------------------------------------- transformasi\n\ndef cek_transformasi(seri: pd.Series, mode: str) -> list[str]:\n    """Galat yang mencegah transformasi dijalankan."""\n    if mode == "log":\n        buruk = seri.dropna() <= 0\n        if buruk.any():\n            contoh = ", ".join(label_periode(p) for p in buruk[buruk].index[:6])\n            return [f"Transformasi log membutuhkan nilai > 0. Ada {int(buruk.sum())} nilai ≤ 0 (misalnya {contoh}). "\n                    "Pilih \'Tanpa transformasi\' atau perbaiki datanya."]\n    return []\n\n\ndef petunjuk_log_ganda(seri: pd.Series, mode: str) -> str | None:\n    """Peringatan bila data tampaknya sudah dalam log tetapi akan di-log lagi."""\n    s = seri.dropna()\n    if mode == "log" and len(s) and s.min() > 0 and s.max() < 30 and (s.max() / s.min()) < 3:\n        return ("Nilai data kecil dan rentangnya sempit (seperti data yang sudah di-log). Jika kolom ini sudah "\n                "berupa log, pilih \'Data sudah dalam bentuk log\' agar tidak di-log dua kali.")\n    return None\n\n\ndef ke_skala_model(seri: pd.Series, mode: str) -> pd.Series:\n    return np.log(seri) if mode == "log" else seri.astype(float)\n\n\ndef ke_skala_asli(x, mode: str):\n    """Kembalikan dari skala model ke skala asli data."""\n    return np.exp(x) if mode in ("log", "sudah_log") else x\n\n\ndef potong(seri: pd.Series, awal: pd.Period, akhir: pd.Period) -> pd.Series:\n    return seri.loc[awal:akhir]\n\n\n# --------------------------------------------------------------------------- variabel independen (X)\n\ndef kolom_x_otomatis(df: pd.DataFrame, kolom_tanggal: str, kolom_target: str) -> list[str]:\n    """Semua kolom selain tanggal dan target yang sebagian besar berisi angka."""\n    hasil = []\n    for k in df.columns:\n        if k in (kolom_tanggal, kolom_target):\n            continue\n        nilai = ke_angka(df[k])\n        terisi = df[k].notna()\n        if terisi.any() and nilai[terisi].notna().mean() > 0.9:\n            hasil.append(k)\n    return hasil\n\n\ndef adalah_dummy(x: pd.Series) -> bool:\n    """True bila nilai (yang terisi) hanya 0 dan/atau 1."""\n    nilai = set(pd.Series(x).dropna().unique().tolist())\n    return bool(nilai) and nilai <= {0, 1, 0.0, 1.0}\n\n\n@dataclass\nclass HasilValidasiX:\n    X: pd.DataFrame\n    galat: list[str] = field(default_factory=list)\n    peringatan: list[str] = field(default_factory=list)\n\n    @property\n    def ok(self) -> bool:\n        return not self.galat\n\n\ndef siapkan_x(df: pd.DataFrame, kolom_tanggal: str, kolom_x: list[str], frekuensi: str,\n              index_target: pd.PeriodIndex) -> HasilValidasiX:\n    """Bentuk tabel X dengan periode yang sama dengan target (termasuk periode masa depan bila ada)."""\n    kode, _ = FREKUENSI[frekuensi]\n    galat, peringatan = [], []\n    df = df.loc[df[kolom_tanggal].notna()]\n    periode = ke_periode(df[kolom_tanggal], kode)\n    data = {}\n    for k in kolom_x:\n        nilai = ke_angka(df[k])\n        tak_terbaca = nilai.isna() & df[k].notna()\n        if tak_terbaca.mean() > 0.5:\n            galat.append(f"Kolom X \'{k}\' sebagian besar bukan angka.")\n            continue\n        if tak_terbaca.any():\n            peringatan.append(f"{int(tak_terbaca.sum())} nilai di kolom X \'{k}\' bukan angka dan dianggap kosong.")\n        data[k] = pd.Series(nilai.to_numpy(), index=periode)\n    X = pd.DataFrame(data)\n    if X.index.duplicated().any():\n        galat.append("Ada periode ganda pada data X; periksa frekuensi data.")\n        return HasilValidasiX(X, galat, peringatan)\n    akhir = max(index_target.max(), X.index.max()) if len(X) else index_target.max()\n    X = X.reindex(pd.period_range(index_target.min(), akhir, freq=index_target.freq))\n    return HasilValidasiX(X, galat, peringatan)\n\n\ndef transformasi_x(X: pd.DataFrame, mode: str) -> tuple[pd.DataFrame, list[str]]:\n    """mode \'tanpa\' atau \'log\'. Dengan \'log\', kolom dummy (0/1) TIDAK di-log dan nama kolom menjadi log(X)."""\n    if mode == "tanpa" or X.empty:\n        return X.astype(float), []\n    hasil, galat = {}, []\n    for k in X.columns:\n        if adalah_dummy(X[k]):\n            hasil[k] = X[k].astype(float)\n        elif (X[k].dropna() <= 0).any():\n            galat.append(f"Kolom X \'{k}\' punya nilai ≤ 0 sehingga tidak bisa di-log; pilih transformasi X \'tanpa\'.")\n        else:\n            hasil[f"log({k})"] = np.log(X[k].astype(float))\n    return pd.DataFrame(hasil, index=X.index), galat\n\n\ndef bentuk_lag(X_model: pd.DataFrame, lags: tuple[int, ...] = (0,)) -> pd.DataFrame:\n    """Matriks desain X: lag 0 memakai nama asli, lag k > 0 bernama X(-k)."""\n    kolom = {}\n    for k in X_model.columns:\n        for lag in sorted(set(lags)):\n            kolom[k if lag == 0 else f"{k}(-{lag})"] = X_model[k].shift(lag)\n    return pd.DataFrame(kolom, index=X_model.index)\n\n\ndef nama_dasar_x(kolom_desain: str) -> str:\n    """\'log(A)(-2)\' -> \'log(A)\'."""\n    return re.sub(r"\\(-\\d+\\)$", "", kolom_desain)\n',
    'modules/exploration.py': '"""Ringkasan eksplorasi data (hanya informasi; tidak memengaruhi model)."""\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom .data_preprocessing import label_periode\n\n\ndef statistik_deskriptif(seri: pd.Series) -> pd.Series:\n    s = seri.dropna()\n    return pd.Series({\n        "Jumlah periode": len(seri),\n        "Observasi terisi": len(s),\n        "Nilai kosong": int(seri.isna().sum()),\n        "Rata-rata": s.mean(),\n        "Median": s.median(),\n        "Simpangan baku": s.std(),\n        "Minimum": s.min(),\n        "Maksimum": s.max(),\n        "Periode awal": label_periode(seri.index.min()),\n        "Periode akhir": label_periode(seri.index.max()),\n    })\n\n\ndef daftar_kosong(seri: pd.Series) -> list[str]:\n    return [label_periode(p) for p in seri.index[seri.isna()]]\n\n\ndef dekomposisi(seri: pd.Series, s: int, model: str = "additive"):\n    """Dekomposisi klasik (moving average). Butuh data tanpa kosong dan ≥ 2 musim penuh."""\n    from statsmodels.tsa.seasonal import seasonal_decompose\n\n    if seri.isna().any():\n        raise ValueError("Dekomposisi tidak bisa dijalankan karena ada nilai kosong.")\n    if len(seri) < 2 * s:\n        raise ValueError(f"Dekomposisi butuh minimal {2 * s} observasi (2 musim penuh).")\n    if model == "multiplicative" and (seri <= 0).any():\n        raise ValueError("Dekomposisi multiplikatif butuh semua nilai > 0.")\n    hasil = seasonal_decompose(seri.to_timestamp(), model=model, period=s)\n    return pd.DataFrame({"Data": hasil.observed, "Tren": hasil.trend, "Musiman": hasil.seasonal,\n                         "Residual": hasil.resid})\n',
    'modules/arima_model.py': '"""\nEstimasi (S)ARIMA / (S)ARIMAX dengan exact maximum likelihood, beserta statistik dan diagnostiknya.\n\nMetodologi (berlaku untuk dataset apa pun):\n\n1. Differencing dilakukan eksplisit sebelum estimasi, pada Y dan pada setiap variabel X:\n       y*_t = (1 - L)^d (1 - L^s)^D y_t,   x*_t = (1 - L)^d (1 - L^s)^D x_t\n   Sampel estimasi = data setelah differencing (d + D·s observasi awal hilang).\n2. Model pada y*: regresi pada konstanta (dan X) dengan galat ARMA\n       y*_t = C + β\'x*_t + u_t,   φ(L) Φ(L^s) u_t = θ(L) Θ(L^s) ε_t\n   sehingga C adalah RATA-RATA y* bersih dari pengaruh X (bila d = D = 0) atau drift (bila ada\n   differencing), bukan intercept trend=\'c\' statsmodels.\n3. Likelihood eksak: state ARMA diinisialisasi dengan distribusi stasionernya, jadi\n   semua observasi ikut dihitung (tidak ada burn-in).\n4. Kovarians koefisien: outer product of gradients (OPG). Uji koefisien memakai distribusi t\n   dengan derajat bebas n - k (k termasuk σ²).\n5. Stasioneritas/invertibilitas tidak dipaksa (bisa diaktifkan lewat opsi).\n6. Residual ε̂_t = inovasi terstandar × σ̂, sehingga Σ ε̂² = n·σ̂². Statistik ringkas (R², adj. R²,\n   S.E. of regression, F, Durbin-Watson) dihitung dari residual ini dan dari y*.\n   Kriteria informasi per observasi: (-2·LL + 2k)/n.\n\nProyeksi memakai model level (dengan differencing di dalam state space) yang parameternya\ndikunci ke hasil estimasi di atas, sehingga ramalan dan selangnya otomatis kembali ke level.\nUntuk SARIMAX, nilai X periode ramalan wajib diberikan (diisi manual atau hasil proyeksi X).\n\nstatsmodels diimpor di dalam fungsi agar modul lain tetap bisa diuji tanpa statsmodels.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass, field, replace\nfrom itertools import product\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\n\n# --------------------------------------------------------------------------- spesifikasi\n\n@dataclass\nclass SpesifikasiModel:\n    order: tuple[int, int, int] = (0, 0, 0)\n    seasonal_order: tuple[int, int, int, int] = (1, 0, 0, 12)\n    konstanta: bool = True\n    paksa_stasioner: bool = False\n    paksa_invertible: bool = False\n    exog: tuple[str, ...] = ()          # nama kolom X (sudah ditransformasi/di-lag) yang ikut model\n\n    def label(self, dengan_x: bool = True) -> str:\n        p, d, q = self.order\n        P, D, Q, s = self.seasonal_order\n        musiman = bool(P or D or Q)\n        nama = ("SARIMA" if musiman else "ARIMA") + ("X" if self.exog else "")\n        teks = f"{nama}({p},{d},{q})" + (f"({P},{D},{Q}){s}" if musiman else "")\n        teks += " + C" if self.konstanta else ""\n        if self.exog and dengan_x:\n            teks += " + " + " + ".join(self.exog)\n        return teks\n\n    def tanpa_x(self) -> "SpesifikasiModel":\n        return replace(self, exog=())\n\n    def jumlah_parameter_arma(self) -> int:\n        p, _, q = self.order\n        P, _, Q, _ = self.seasonal_order\n        return p + q + P + Q\n\n    def musiman_statsmodels(self, tanpa_differencing: bool = False) -> tuple[int, int, int, int]:\n        """seasonal_order untuk statsmodels; (0,0,0,0) bila tidak ada komponen musiman."""\n        P, D, Q, s = self.seasonal_order\n        if tanpa_differencing:\n            D = 0\n        return (P, D, Q, s) if (P or D or Q) else (0, 0, 0, 0)\n\n    def observasi_hilang(self) -> int:\n        """Jumlah observasi awal yang hilang karena differencing."""\n        d = self.order[1]\n        _, D, _, s = self.seasonal_order\n        return d + D * s\n\n    def minimal_observasi(self) -> int:\n        """Batas bawah kasar agar model bisa diestimasi."""\n        p, _, q = self.order\n        P, _, Q, s = self.seasonal_order\n        lag_maks = max(p, P * s if P else 0, q, Q * s if Q else 0)\n        return max(10, self.observasi_hilang() + lag_maks + self.jumlah_parameter_arma() + len(self.exog) + 5)\n\n\n# --------------------------------------------------------------------------- utilitas\n\ndef diferensiasi(y, spek: SpesifikasiModel):\n    """(1 - L)^d (1 - L^s)^D untuk Series atau DataFrame, tanpa observasi awal yang hilang."""\n    d = spek.order[1]\n    _, D, _, s = spek.seasonal_order\n    hasil = y.astype(float)\n    for _ in range(D):\n        hasil = hasil - hasil.shift(s)\n    for _ in range(d):\n        hasil = hasil - hasil.shift(1)\n    return hasil.iloc[spek.observasi_hilang():]\n\n\ndef regresor_konstanta(n: int, spek: SpesifikasiModel) -> np.ndarray:\n    """Regresor x (level) yang setelah differencing menjadi deret 1.\n\n    Dipakai di model level agar koefisiennya sama dengan C pada data hasil differencing.\n    """\n    d = spek.order[1]\n    _, D, _, s = spek.seasonal_order\n    x = np.ones(n)\n    for _ in range(D):\n        for t in range(s, n):\n            x[t] += x[t - s]\n    for _ in range(d):\n        x = np.cumsum(x)\n    return x\n\n\ndef matriks_exog(index, konstanta_nilai, X: pd.DataFrame | None, spek: SpesifikasiModel):\n    """Gabungan kolom C (bila dipakai) dan kolom X, urut seperti spek.exog. None bila kosong."""\n    bagian = []\n    if spek.konstanta:\n        bagian.append(pd.DataFrame({"C": konstanta_nilai}, index=index))\n    if spek.exog:\n        bagian.append(X.loc[index, list(spek.exog)].astype(float))\n    return pd.concat(bagian, axis=1) if bagian else None\n\n\ndef _kelas_sarimax():\n    """SARIMAX dengan inisialisasi eksak: state differencing difus, state ARMA stasioner."""\n    from statsmodels.tsa.statespace.initialization import Initialization\n    from statsmodels.tsa.statespace.sarimax import SARIMAX\n\n    class SARIMAXEksak(SARIMAX):\n        def initialize_default(self, approximate_diffuse_variance=None):\n            if approximate_diffuse_variance is None:\n                approximate_diffuse_variance = self.ssm.initial_variance\n            init = Initialization(self.k_states, approximate_diffuse_variance=approximate_diffuse_variance)\n            k_diff = self._k_states_diff\n            if k_diff:\n                init.set((0, k_diff), "diffuse")\n            if self.k_states > k_diff:\n                init.set((k_diff, self.k_states), "stationary")\n            self.ssm.initialization = init\n\n    return SARIMAXEksak\n\n\ndef selaraskan_x(y_model: pd.Series, X: pd.DataFrame | None, spek: SpesifikasiModel):\n    """Potong awal sampel bila X kosong di awal (mis. akibat lag) dan periksa X lengkap.\n\n    Mengembalikan (y_model, X) yang sudah selaras, plus catatan.\n    """\n    if not spek.exog:\n        return y_model, None, []\n    if X is None:\n        raise ValueError("Model memakai variabel X, tetapi data X tidak diberikan.")\n    kurang = [k for k in spek.exog if k not in X.columns]\n    if kurang:\n        raise ValueError(f"Kolom X tidak ditemukan: {\', \'.join(kurang)}.")\n    Xs = X.reindex(y_model.index)[list(spek.exog)].astype(float)\n    lengkap = Xs.notna().all(axis=1)\n    if not lengkap.any():\n        raise ValueError("Variabel X kosong di seluruh periode estimasi.")\n    awal = lengkap.idxmax()\n    catatan = []\n    if awal != y_model.index[0]:\n        catatan.append(f"Sampel estimasi dimulai {awal} karena X belum tersedia sebelumnya (mis. akibat lag X).")\n    y_model, Xs = y_model.loc[awal:], Xs.loc[awal:]\n    if Xs.isna().any().any():\n        kosong = Xs.index[Xs.isna().any(axis=1)]\n        raise ValueError(f"Variabel X kosong di dalam periode estimasi ({\', \'.join(str(p) for p in kosong[:6])}). "\n                         "Lengkapi data X atau persempit periode.")\n    return y_model, Xs, catatan\n\n\n# --------------------------------------------------------------------------- estimasi\n\n@dataclass\nclass ModelTerestimasi:\n    """Hasil estimasi satu model.\n\n    res      : hasil statsmodels pada data setelah differencing (koefisien, LL, residual)\n    spek     : spesifikasi model\n    y_model  : data skala model (mis. log) SEBELUM differencing, dipakai untuk proyeksi\n    X        : matriks X (skala model, sebelum differencing) selaras dengan y_model, atau None\n    catatan  : catatan proses estimasi (mis. fallback ke batasan stasioner)\n    """\n    res: object\n    spek: SpesifikasiModel\n    y_model: pd.Series\n    X: pd.DataFrame | None = None\n    catatan: list[str] = field(default_factory=list)\n\n    @property\n    def params(self) -> pd.Series:\n        return self.res.params\n\n    @property\n    def bse(self) -> pd.Series:\n        return self.res.bse\n\n    @property\n    def llf(self) -> float:\n        return float(self.res.llf)\n\n    @property\n    def nobs(self) -> int:\n        return int(self.res.nobs)\n\n    @property\n    def df_resid(self) -> int:\n        return max(self.nobs - len(self.params), 1)\n\n    @property\n    def resid(self) -> pd.Series:\n        """Residual ε̂_t = inovasi terstandar × σ̂ (Σ ε̂² = n·σ̂²)."""\n        std = np.asarray(self.res.standardized_forecasts_error)[0]\n        sigma = float(np.sqrt(self.res.params["sigma2"]))\n        return pd.Series(std * sigma, index=self.y_estimasi.index).dropna()\n\n    @property\n    def y_estimasi(self) -> pd.Series:\n        """Variabel dependen yang benar-benar diestimasi (setelah differencing)."""\n        return diferensiasi(self.y_model, self.spek)\n\n    def summary(self):\n        return self.res.summary()\n\n    def model_level(self):\n        """Model level dengan parameter terkunci, untuk ramalan dalam skala model."""\n        spek = self.spek\n        kelas = _kelas_sarimax()\n        x = regresor_konstanta(len(self.y_model), spek)\n        model = kelas(\n            self.y_model.astype(float),\n            exog=matriks_exog(self.y_model.index, x, self.X, spek),\n            order=spek.order, seasonal_order=spek.musiman_statsmodels(), trend="n",\n            enforce_stationarity=False, enforce_invertibility=False,\n        )\n        return model.filter(np.asarray(self.res.params))\n\n\ndef _fit(y_est: pd.Series, X_est: pd.DataFrame | None, spek: SpesifikasiModel, paksa_stasioner: bool):\n    p, _, q = spek.order\n    kelas = _kelas_sarimax()\n    model = kelas(\n        y_est,\n        exog=matriks_exog(y_est.index, np.ones(len(y_est)), X_est, spek),\n        order=(p, 0, q), seasonal_order=spek.musiman_statsmodels(tanpa_differencing=True), trend="n",\n        enforce_stationarity=paksa_stasioner, enforce_invertibility=spek.paksa_invertible,\n    )\n    return model.fit(disp=False, maxiter=2000, cov_type="opg")\n\n\ndef _stasioner(res) -> bool:\n    akar = np.asarray(res.arroots) if len(res.arroots) else np.array([])\n    return bool(np.all(np.abs(akar) > 1)) if len(akar) else True\n\n\ndef estimasi(y_model: pd.Series, spek: SpesifikasiModel, X: pd.DataFrame | None = None) -> ModelTerestimasi:\n    """Estimasi model pada data skala model (mis. log), sebelum differencing.\n\n    X: DataFrame variabel independen dalam skala model (indeks periode, boleh lebih panjang dari\n       y_model); hanya kolom spek.exog yang dipakai.\n    """\n    import warnings\n\n    y_model = y_model.astype(float)\n    y_model, X_sel, catatan = selaraskan_x(y_model, X, spek)\n    y_est = diferensiasi(y_model, spek)\n    X_est = diferensiasi(X_sel, spek) if X_sel is not None else None\n    res = None\n    if not spek.paksa_stasioner:\n        try:\n            with warnings.catch_warnings():\n                warnings.simplefilter("ignore")\n                res = _fit(y_est, X_est, spek, paksa_stasioner=False)\n            if not (np.isfinite(res.llf) and _stasioner(res)):\n                res = None\n        except Exception:\n            res = None\n        if res is None:\n            catatan.append("Estimasi tanpa batasan keluar dari wilayah stasioner; model diestimasi ulang dengan "\n                           "batasan stasioner.")\n    if res is None:\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore")\n            res = _fit(y_est, X_est, spek, paksa_stasioner=True)\n    return ModelTerestimasi(res=res, spek=spek, y_model=y_model, X=X_sel, catatan=catatan)\n\n\ndef konvergen(model: ModelTerestimasi) -> bool:\n    retvals = getattr(model.res, "mle_retvals", None) or {}\n    return bool(retvals.get("converged", True))\n\n\n# --------------------------------------------------------------------------- pemilihan orde otomatis\n\ndef pilih_orde(y_model: pd.Series, s: int, X: pd.DataFrame | None = None, exog: tuple[str, ...] = (),\n               konstanta: bool = True, d: int = 0, D: int = 0, maks_p: int = 2, maks_q: int = 2,\n               maks_P: int = 1, maks_Q: int = 1, kriteria: str = "AIC") -> tuple[pd.DataFrame, list]:\n    """Estimasi semua kombinasi (p,q)(P,Q) hingga batas maksimum, urutkan menurut kriteria informasi.\n\n    Semua model memakai sampel yang sama (exact ML), sehingga kriterianya sebanding.\n    Mengembalikan (tabel, daftar SpesifikasiModel urut terbaik).\n    """\n    kolom_kriteria = {"AIC": "Akaike info criterion (AIC)", "SC": "Schwarz criterion (SC/BIC)",\n                      "HQ": "Hannan-Quinn criterion (HQ)"}[kriteria]\n    musiman = s >= 2\n    baris, spek_list = [], []\n    for p, q, P, Q in product(range(maks_p + 1), range(maks_q + 1),\n                              range(maks_P + 1 if musiman else 1), range(maks_Q + 1 if musiman else 1)):\n        spek = SpesifikasiModel(order=(p, d, q), seasonal_order=(P, D, Q, s if (musiman or D) else 0),\n                                konstanta=konstanta, exog=tuple(exog))\n        try:\n            m = estimasi(y_model, spek, X)\n            st = statistik_ringkas(m)\n            baris.append({"ARMA (p,q)(P,Q)": f"({p},{q})({P},{Q})", "Model": spek.label(dengan_x=False),\n                          "AIC": st["Akaike info criterion (AIC)"], "SC": st["Schwarz criterion (SC/BIC)"],\n                          "HQ": st["Hannan-Quinn criterion (HQ)"], "Log likelihood": st["Log likelihood"],\n                          "Konvergen": konvergen(m), "Batasan stasioner": bool(m.catatan),\n                          "_urut": st[kolom_kriteria]})\n        except Exception as e:  # model gagal diestimasi tetap dilaporkan\n            baris.append({"ARMA (p,q)(P,Q)": f"({p},{q})({P},{Q})", "Model": spek.label(dengan_x=False),\n                          "AIC": np.nan, "SC": np.nan, "HQ": np.nan, "Log likelihood": np.nan,\n                          "Konvergen": False, "Batasan stasioner": False, "_urut": np.inf,\n                          "Catatan": str(e)[:80]})\n        spek_list.append(spek)\n    tabel = pd.DataFrame(baris)\n    urutan = tabel["_urut"].argsort(kind="stable").to_numpy()\n    tabel = tabel.iloc[urutan].drop(columns="_urut").reset_index(drop=True)\n    tabel.index = tabel.index + 1\n    return tabel, [spek_list[i] for i in urutan]\n\n\n# --------------------------------------------------------------------------- tabel & statistik\n\ndef nama_parameter(nama: str) -> str:\n    """Nama parameter statsmodels -> notasi ringkas (ar.S.L12 -> SAR(12), sigma2 -> SIGMASQ)."""\n    if nama == "sigma2":\n        return "SIGMASQ"\n    m = re.match(r"^(ar|ma)\\.(S\\.)?L(\\d+)$", nama)\n    if m:\n        jenis, musiman, lag = m.groups()\n        return f"{\'S\' if musiman else \'\'}{jenis.upper()}({lag})"\n    return nama\n\n\ndef _urutan_parameter(nama: str) -> int:\n    """C, X, AR, SAR, MA, SMA, SIGMASQ."""\n    if nama == "sigma2":\n        return 9\n    for i, awal in enumerate(("ar.L", "ar.S.", "ma.L", "ma.S."), start=2):\n        if nama.startswith(awal):\n            return i\n    return 0 if nama == "C" else 1\n\n\ndef tabel_koefisien(model: ModelTerestimasi) -> pd.DataFrame:\n    res = model.res\n    t = res.params / res.bse\n    tabel = pd.DataFrame({\n        "Koefisien": res.params,\n        "Std. Error": res.bse,\n        "t-Statistic": t,\n        "Prob.": 2 * stats.t.sf(np.abs(t), model.df_resid),\n    })\n    tabel = tabel.iloc[sorted(range(len(tabel)), key=lambda i: (_urutan_parameter(tabel.index[i]), i))]\n    tabel.index = [nama_parameter(n) for n in tabel.index]\n    tabel.index.name = "Variabel"\n    return tabel\n\n\ndef statistik_ringkas(model: ModelTerestimasi) -> pd.Series:\n    n = model.nobs\n    k = len(model.params)  # termasuk SIGMASQ\n    ll = model.llf\n    e = model.resid\n    y = model.y_estimasi.reindex(e.index)\n    ssr = float((e ** 2).sum())\n    sst = float(((y - y.mean()) ** 2).sum())\n    r2 = 1 - ssr / sst if sst > 0 else np.nan\n    df = n - k\n    if k > 1 and df > 0 and np.isfinite(r2) and r2 < 1:\n        f_stat = (r2 / (k - 1)) / ((1 - r2) / df)\n        prob_f = float(stats.f.sf(f_stat, k - 1, df))\n    else:\n        f_stat = prob_f = np.nan\n    return pd.Series({\n        "Jumlah observasi": n,\n        "Observasi hilang karena differencing": model.spek.observasi_hilang(),\n        "R-squared": r2,\n        "Adjusted R-squared": 1 - (1 - r2) * (n - 1) / df if df > 0 else np.nan,\n        "S.E. of regression": float(np.sqrt(ssr / df)) if df > 0 else np.nan,\n        "Sum squared resid": ssr,\n        "Log likelihood": ll,\n        "F-statistic": f_stat,\n        "Prob(F-statistic)": prob_f,\n        "Mean dependent var": float(y.mean()),\n        "S.D. dependent var": float(y.std(ddof=1)),\n        "Akaike info criterion (AIC)": (-2 * ll + 2 * k) / n,\n        "Schwarz criterion (SC/BIC)": (-2 * ll + k * np.log(n)) / n,\n        "Hannan-Quinn criterion (HQ)": (-2 * ll + 2 * k * np.log(np.log(n))) / n,\n        "Durbin-Watson stat": float((np.diff(e.to_numpy()) ** 2).sum() / ssr) if ssr > 0 else np.nan,\n    })\n\n\ndef diagnostik(model: ModelTerestimasi) -> dict:\n    from statsmodels.stats.diagnostic import acorr_ljungbox\n    from statsmodels.tsa.stattools import acf\n\n    res, spek = model.res, model.spek\n    resid = model.resid\n    s = spek.seasonal_order[3]\n    df_model = spek.jumlah_parameter_arma()\n    kandidat = {s, 2 * s, 12, 24} if s > 1 else {6, 12, 24}\n    lag_uji = sorted({lag for lag in kandidat if df_model < lag < len(resid)})\n    ljung_box = acorr_ljungbox(resid, lags=lag_uji, model_df=df_model) if lag_uji else pd.DataFrame()\n\n    jb = stats.jarque_bera(resid)\n    nlags = max(1, min(max(2 * s, 12), 36, len(resid) - 1))\n    nilai_acf = acf(resid, nlags=nlags, fft=False, missing="drop")[1:]\n\n    akar_ar = 1 / np.abs(res.arroots) if len(res.arroots) else np.array([])\n    akar_ma = 1 / np.abs(res.maroots) if len(res.maroots) else np.array([])\n\n    return {\n        "ljung_box": ljung_box,\n        "jarque_bera": {"statistik": float(jb.statistic), "p_value": float(jb.pvalue)},\n        "acf": pd.DataFrame({"lag": np.arange(1, nlags + 1), "acf": nilai_acf}),\n        "batas_acf": 1.96 / np.sqrt(len(resid)),\n        "akar_ar_invers_maks": float(akar_ar.max()) if len(akar_ar) else None,\n        "akar_ma_invers_maks": float(akar_ma.max()) if len(akar_ma) else None,\n        "peringatan": list(model.catatan) + peringatan_model(res.params, res.bse, akar_ar, akar_ma, s),\n    }\n\n\ndef peringatan_model(params: pd.Series, bse: pd.Series, akar_ar, akar_ma, s: int,\n                     ambang: float = 0.99) -> list[str]:\n    """Deteksi model yang tidak bisa ditafsirkan seperti ARIMA biasa."""\n    pesan = []\n    if len(akar_ar) and np.max(akar_ar) >= ambang:\n        pesan.append(f"Akar AR invers = {np.max(akar_ar):.4f} (≥ {ambang}): komponen AR di/dekat unit root, "\n                     "model tidak stasioner.")\n    if len(akar_ma) and np.max(akar_ma) >= ambang:\n        pesan.append(f"Akar MA invers = {np.max(akar_ma):.4f} (≥ {ambang}): komponen MA tidak/nyaris tidak invertible.")\n    for ar, ma in ((f"ar.S.L{s}", f"ma.S.L{s}"), ("ar.L1", "ma.L1")):\n        if ar in params.index and ma in params.index:\n            phi, theta = params[ar], params[ma]\n            if abs(phi + theta) < 0.05 and abs(phi) > 0.5:\n                pesan.append(f"{nama_parameter(ar)} = {phi:.4f} dan {nama_parameter(ma)} = {theta:.4f} hampir saling "\n                             "meniadakan (pembatalan akar); t-statistik yang sangat besar tidak bermakna.")\n    if (bse.drop("sigma2", errors="ignore") < 1e-3).any():\n        pesan.append("Ada standard error sangat kecil (< 0,001): estimasi berada di batas ruang parameter.")\n    return pesan\n\n\ndef ringkasan_teks(model: ModelTerestimasi, stat: pd.Series) -> str:\n    """Ringkasan model sebagai teks (untuk diunduh)."""\n    garis = [f"Model: {model.spek.label()}", ""]\n    garis.append(tabel_koefisien(model).to_string(float_format=lambda v: f"{v:.6f}"))\n    garis += ["", stat.to_string(float_format=lambda v: f"{v:.6f}"), "", "--- statsmodels summary ---",\n              str(model.summary())]\n    return "\\n".join(garis)\n',
    'modules/forecasting.py': '"""\nBacktest dan proyeksi.\n\n- Pembagian latih/uji kronologis (tidak diacak).\n- Backtest: model diestimasi HANYA dengan data latih, lalu diramal dinamis multi-langkah\n  sepanjang periode uji (tidak memakai nilai Y aktual periode uji).\n- Untuk model dengan X (SARIMAX) ada dua jenis ramalan:\n    * ex-post : nilai X periode ramalan dianggap diketahui (X aktual / X diisi manual);\n    * ex-ante : nilai X periode ramalan sendiri diproyeksikan dengan SARIMA dari data X sebelumnya.\n- Proyeksi ke depan: model diestimasi ulang dengan data s.d. observasi terakhir.\n- Ramalan dibuat di level (differencing ditangani model), lalu dikembalikan ke skala asli\n  dengan exp() bila model memakai log.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import arima_model as am\nfrom .data_preprocessing import adalah_dummy, bentuk_lag, ke_skala_asli\n\n\ndef proyeksi(model: am.ModelTerestimasi, langkah: int, mode_transformasi: str,\n             alpha: float = 0.05, koreksi_bias: bool = False,\n             X_depan: pd.DataFrame | None = None) -> pd.DataFrame:\n    """Ramalan dinamis sebanyak langkah periode setelah akhir data estimasi, dalam skala asli.\n\n    X_depan: nilai X (skala model, kolom = spek.exog) untuk periode ramalan; wajib untuk SARIMAX.\n    """\n    spek = model.spek\n    y = model.y_model\n    index_depan = pd.period_range(y.index[-1] + 1, periods=langkah, freq=y.index.freq)\n    if spek.exog:\n        if X_depan is None:\n            raise ValueError("Model memakai X: nilai X untuk periode ramalan wajib diberikan.")\n        X_depan = X_depan.reindex(index_depan)[list(spek.exog)]\n        if X_depan.isna().any().any():\n            kosong = X_depan.index[X_depan.isna().any(axis=1)]\n            raise ValueError(f"Nilai X untuk periode ramalan belum lengkap ({\', \'.join(str(p) for p in kosong[:6])}).")\n    x_c = am.regresor_konstanta(len(y) + langkah, spek)[len(y):]\n    exog_depan = am.matriks_exog(index_depan, x_c, X_depan, spek)\n\n    ramalan = model.model_level().get_forecast(steps=langkah, exog=exog_depan)\n    rata = np.asarray(ramalan.predicted_mean)\n    se = np.asarray(ramalan.se_mean)\n    ci = np.asarray(ramalan.conf_int(alpha=alpha))\n    pakai_exp = mode_transformasi in ("log", "sudah_log")\n    faktor = np.exp(se ** 2 / 2) if (koreksi_bias and pakai_exp) else 1.0\n    return pd.DataFrame({\n        "ramalan_skala_model": rata,\n        "se_skala_model": se,\n        "ramalan": ke_skala_asli(rata, mode_transformasi) * faktor,\n        "batas_bawah": ke_skala_asli(ci[:, 0], mode_transformasi),\n        "batas_atas": ke_skala_asli(ci[:, 1], mode_transformasi),\n    }, index=index_depan)\n\n\n# --------------------------------------------------------------------------- proyeksi variabel X\n\ndef ramal_x_sarima(X_model: pd.DataFrame, akhir: pd.Period, langkah: int, s: int,\n                   maks_orde: tuple[int, int, int, int] = (1, 1, 1, 1)) -> tuple[pd.DataFrame, dict]:\n    """Proyeksikan setiap kolom X (skala model, tanpa lag) dengan SARIMA + C.\n\n    Orde tiap variabel dipilih otomatis (AIC terkecil) dari data X s.d. akhir.\n    Mengembalikan (DataFrame ramalan X untuk langkah periode setelah akhir, info orde per kolom).\n    """\n    index_depan = pd.period_range(akhir + 1, periods=langkah, freq=X_model.index.freq)\n    hasil, info = {}, {}\n    for k in X_model.columns:\n        x = X_model[k].loc[:akhir].dropna()\n        if x.nunique() <= 1:  # konstan: ramalan = nilai itu\n            hasil[k] = np.full(langkah, x.iloc[-1] if len(x) else np.nan)\n            info[k] = "konstan"\n            continue\n        maks_p, maks_q, maks_P, maks_Q = maks_orde\n        tabel, daftar = am.pilih_orde(x, s, maks_p=maks_p, maks_q=maks_q, maks_P=maks_P, maks_Q=maks_Q)\n        terbaik = next((sp for sp, ok in zip(daftar, tabel["Konvergen"]) if ok), daftar[0])\n        m = am.estimasi(x, terbaik)\n        hasil[k] = proyeksi(m, langkah, "tanpa")["ramalan"].to_numpy()\n        info[k] = terbaik.label() + (" — dummy diproyeksi SARIMA; nilai kalender sebaiknya diisi manual"\n                                     if adalah_dummy(x) else "")\n    return pd.DataFrame(hasil, index=index_depan), info\n\n\ndef desain_x_depan(X_model: pd.DataFrame, X_ramalan: pd.DataFrame, lags: tuple[int, ...]) -> pd.DataFrame:\n    """Gabungkan X historis + X periode ramalan, lalu bentuk kolom lag untuk periode ramalan."""\n    gabung = pd.concat([X_model.loc[:X_ramalan.index[0] - 1], X_ramalan])\n    return bentuk_lag(gabung, lags).loc[X_ramalan.index]\n\n\n# --------------------------------------------------------------------------- baseline\n\ndef baseline_naif_musiman(y_model_latih: pd.Series, periode_uji: pd.PeriodIndex, s: int) -> pd.Series:\n    """Ramalan = nilai periode yang sama satu musim sebelumnya (berulang bila horizon > s)."""\n    nilai = y_model_latih.copy()\n    for p in periode_uji:\n        nilai.loc[p] = nilai.get(p - s, np.nan)\n    return nilai.loc[periode_uji]\n\n\ndef baseline_rata_musiman(y_model_latih: pd.Series, periode_uji: pd.PeriodIndex, s: int) -> pd.Series:\n    """Ramalan = rata-rata (di skala model) per posisi musim pada data latih."""\n    y = y_model_latih.dropna()\n    posisi = pd.Series([p.ordinal % s for p in y.index], index=y.index)\n    rata = y.groupby(posisi).mean()\n    return pd.Series([rata.get(p.ordinal % s, np.nan) for p in periode_uji], index=periode_uji)\n\n\n# --------------------------------------------------------------------------- backtest\n\ndef _tabel(aktual: pd.Series, ramal: pd.DataFrame) -> pd.DataFrame:\n    return pd.DataFrame({\n        "aktual": aktual,\n        "ramalan": ramal["ramalan"],\n        "galat": aktual - ramal["ramalan"],\n        "batas_bawah": ramal["batas_bawah"],\n        "batas_atas": ramal["batas_atas"],\n    })\n\n\ndef backtest(y_model: pd.Series, spek: am.SpesifikasiModel, awal_latih, akhir_latih, akhir_uji,\n             mode_transformasi: str, alpha: float = 0.05, koreksi_bias: bool = False,\n             X: pd.DataFrame | None = None, X_uji_lain: dict[str, pd.DataFrame] | None = None) -> dict:\n    """Estimasi pada data latih, ramal periode uji.\n\n    X          : desain X lengkap (skala model). Untuk SARIMAX, ramalan utama memakai X aktual\n                 periode uji (ex-post).\n    X_uji_lain : skenario X lain untuk periode uji, mis. {"ex-ante": X hasil proyeksi}.\n    """\n    s = spek.seasonal_order[3]\n    y_latih = y_model.loc[awal_latih:akhir_latih]\n    y_uji = y_model.loc[akhir_latih + 1:akhir_uji]\n    model = am.estimasi(y_latih, spek, X)\n    aktual = ke_skala_asli(y_uji, mode_transformasi)\n    X_uji = X.reindex(y_uji.index) if (spek.exog and X is not None) else None\n    tabel = _tabel(aktual, proyeksi(model, len(y_uji), mode_transformasi, alpha, koreksi_bias, X_uji))\n    varian = {}\n    for nama, Xs in (X_uji_lain or {}).items():\n        varian[nama] = _tabel(aktual, proyeksi(model, len(y_uji), mode_transformasi, alpha, koreksi_bias, Xs))\n    y_latih_efektif = model.y_model\n    pembanding = {}\n    if s > 1:\n        pembanding["Naif musiman"] = ke_skala_asli(baseline_naif_musiman(y_latih_efektif, y_uji.index, s),\n                                                   mode_transformasi)\n        pembanding["Rata-rata musiman"] = ke_skala_asli(baseline_rata_musiman(y_latih_efektif, y_uji.index, s),\n                                                       mode_transformasi)\n    pembanding["Naif (nilai terakhir)"] = pd.Series(\n        ke_skala_asli(y_latih_efektif.dropna().iloc[-1], mode_transformasi), index=y_uji.index)\n    return {"hasil": model, "y_latih": y_latih_efektif, "tabel": tabel, "tabel_varian": varian,\n            "pembanding": pembanding}\n',
    'modules/evaluation.py': '"""\nUkuran akurasi ramalan:\n\nMAE   = rata-rata |aktual − ramalan|\nRMSE  = akar rata-rata (aktual − ramalan)²\nMAPE  = rata-rata |aktual − ramalan| / |aktual| × 100\nsMAPE = rata-rata |aktual − ramalan| / ((|aktual| + |ramalan|)/2) × 100\nTheil U2 = akar[ Σ((ramalan_t+1 − aktual_t+1)/aktual_t)² / Σ((aktual_t+1 − aktual_t)/aktual_t)² ],\n           dihitung di dalam periode uji.\n\nPeriode dengan aktual = 0 dikecualikan dari MAPE (pembagian dengan nol) dan jumlahnya\ndilaporkan, bukan disembunyikan.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nPENJELASAN = {\n    "MAE": "Rata-rata selisih absolut antara nilai aktual dan ramalan, dalam satuan data. Makin kecil makin baik.",\n    "RMSE": "Seperti MAE tetapi memberi bobot lebih besar pada kesalahan besar. Satuan sama dengan data.",\n    "MAPE (%)": "Rata-rata kesalahan dalam persen dari nilai aktual. Mudah dibaca, tetapi membengkak bila ada nilai aktual yang sangat kecil.",\n    "sMAPE (%)": "Versi simetris MAPE (dibagi rata-rata aktual dan ramalan), lebih stabil saat nilai aktual kecil.",\n    "Theil U2": "Perbandingan dengan ramalan naif \'sama dengan periode sebelumnya\'. < 1 berarti lebih baik dari naif.",\n}\n\n\ndef ukuran_akurasi(aktual, ramalan) -> dict:\n    a = np.asarray(aktual, dtype=float)\n    f = np.asarray(ramalan, dtype=float)\n    sah = ~(np.isnan(a) | np.isnan(f))\n    a, f = a[sah], f[sah]\n    if len(a) == 0:\n        return {"n": 0}\n    e = a - f\n    bukan_nol = a != 0\n    hasil = {\n        "n": int(len(a)),\n        "MAE": float(np.mean(np.abs(e))),\n        "RMSE": float(np.sqrt(np.mean(e ** 2))),\n        "MAPE (%)": float(np.mean(np.abs(e[bukan_nol] / a[bukan_nol])) * 100) if bukan_nol.any() else np.nan,\n        "Aktual = 0 (dikecualikan dari MAPE)": int((~bukan_nol).sum()),\n    }\n    penyebut = (np.abs(a) + np.abs(f)) / 2\n    ok = penyebut != 0\n    hasil["sMAPE (%)"] = float(np.mean(np.abs(e[ok]) / penyebut[ok]) * 100) if ok.any() else np.nan\n    if len(a) > 1:\n        dasar = a[:-1]\n        ok = dasar != 0\n        pemb = np.sum(((f[1:][ok] - a[1:][ok]) / dasar[ok]) ** 2)\n        peny = np.sum(((a[1:][ok] - dasar[ok]) / dasar[ok]) ** 2)\n        hasil["Theil U2"] = float(np.sqrt(pemb / peny)) if peny > 0 else np.nan\n    return hasil\n\n\ndef tabel_akurasi(aktual: pd.Series, ramalan: dict[str, pd.Series]) -> pd.DataFrame:\n    urutan = ["MAE", "RMSE", "MAPE (%)", "sMAPE (%)", "Theil U2", "n", "Aktual = 0 (dikecualikan dari MAPE)"]\n    tabel = pd.DataFrame({nama: ukuran_akurasi(aktual, f.reindex(aktual.index)) for nama, f in ramalan.items()}).T\n    return tabel[[c for c in urutan if c in tabel.columns]]\n',
    'modules/pipeline.py': '"""Satu pintu untuk menjalankan seluruh alur: validasi → backtest → proyeksi → evaluasi.\n\nTanpa X  : satu model (S)ARIMA.\nDengan X : model (S)ARIMAX dibandingkan langsung dengan model yang sama TANPA X, pada sampel,\n           periode latih/uji, dan orde ARMA yang sama. Ramalan SARIMAX ditampilkan dua versi:\n           ex-post (X aktual / X diisi manual) dan ex-ante (X diproyeksi SARIMA).\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field, replace\n\nimport pandas as pd\n\nfrom . import arima_model as am\nfrom . import evaluation as ev\nfrom . import forecasting as fc\nfrom .data_preprocessing import (bentuk_lag, cek_transformasi, ke_skala_asli, ke_skala_model, label_periode,\n                                 transformasi_x)\n\nNAMA_EX_POST_UJI = "ex-post (X aktual)"\nNAMA_EX_ANTE = "ex-ante (X diproyeksi SARIMA)"\nNAMA_EX_POST_DEPAN = "ex-post (X diisi manual)"\nNAMA_TANPA_X = "tanpa X"\n\n\n@dataclass\nclass Pengaturan:\n    spek: am.SpesifikasiModel          # orde model; kolom X diatur lewat kolom_x\n    transformasi: str                  # "log" | "sudah_log" | "tanpa"\n    awal_latih: pd.Period\n    akhir_latih: pd.Period\n    akhir_uji: pd.Period | None        # None = tanpa periode uji\n    horizon: int = 12\n    tingkat_kepercayaan: int = 95\n    koreksi_bias: bool = False\n    # --- variabel independen (SARIMAX)\n    kolom_x: tuple[str, ...] = ()\n    transformasi_x: str = "tanpa"      # "tanpa" | "log" (kolom dummy 0/1 tidak di-log)\n    lag_x: tuple[int, ...] = (0,)\n    proyeksi_x_sarima: bool = True     # proyeksikan X dengan SARIMA (ex-ante)\n    maks_orde_x: tuple[int, int, int, int] = (1, 1, 1, 1)\n\n\n@dataclass\nclass HasilProyeksi:\n    pengaturan: Pengaturan\n    y_asli: pd.Series\n    backtest: dict | None\n    akurasi: pd.DataFrame | None\n    hasil_akhir: am.ModelTerestimasi\n    koefisien_akhir: pd.DataFrame\n    statistik_akhir: pd.Series\n    diagnostik_akhir: dict\n    ramalan: pd.DataFrame\n    akhir_data_proyeksi: pd.Period\n    peringatan: list[str] = field(default_factory=list)\n    # --- khusus SARIMAX\n    ramalan_varian: dict = field(default_factory=dict)     # nama -> DataFrame ramalan ke depan\n    nama_ramalan_utama: str = ""\n    hasil_tanpa_x: am.ModelTerestimasi | None = None\n    backtest_tanpa_x: dict | None = None\n    info_x: dict = field(default_factory=dict)             # orde SARIMA tiap X, kolom desain\n    X_model: pd.DataFrame | None = None                     # X skala model (tanpa lag)\n\n    @property\n    def spek(self) -> am.SpesifikasiModel:\n        return self.hasil_akhir.spek\n\n    @property\n    def pakai_x(self) -> bool:\n        return bool(self.spek.exog)\n\n\nclass GalatPengaturan(Exception):\n    pass\n\n\ndef panjang_uji_bawaan(n_terisi: int, s: int) -> int:\n    """Panjang periode uji bawaan: satu musim penuh bila data cukup panjang, selain itu ±20% data."""\n    if s > 1 and n_terisi > 3 * s:\n        return s\n    return max(1, n_terisi // 5)\n\n\ndef periode_otomatis(seri: pd.Series, s: int, pakai_uji: bool = True) -> tuple:\n    """(awal_latih, akhir_latih, akhir_uji) bawaan dari data itu sendiri, tanpa tanggal tetap."""\n    terisi = seri.dropna().index\n    awal = terisi[0]\n    if not pakai_uji:\n        return awal, terisi[-1], None\n    n_uji = panjang_uji_bawaan(len(terisi), s)\n    return awal, terisi[-1 - n_uji], terisi[-1]\n\n\ndef siapkan_desain_x(X_dasar: pd.DataFrame | None, peng: Pengaturan):\n    """(X_model tanpa lag, X_desain dengan lag, galat)."""\n    if not peng.kolom_x:\n        return None, None, []\n    if X_dasar is None:\n        return None, None, ["Kolom X dipilih tetapi data X tidak diberikan."]\n    kurang = [k for k in peng.kolom_x if k not in X_dasar.columns]\n    if kurang:\n        return None, None, [f"Kolom X tidak ditemukan: {\', \'.join(kurang)}."]\n    X_model, galat = transformasi_x(X_dasar[list(peng.kolom_x)], peng.transformasi_x)\n    return X_model, bentuk_lag(X_model, peng.lag_x), galat\n\n\ndef periksa(seri: pd.Series, peng: Pengaturan, X_dasar: pd.DataFrame | None = None) -> list[str]:\n    galat = list(cek_transformasi(seri.loc[peng.awal_latih:], peng.transformasi))\n    if peng.akhir_latih < peng.awal_latih:\n        galat.append("Akhir data latih harus setelah awal data latih.")\n    if peng.akhir_uji is not None and peng.akhir_uji <= peng.akhir_latih:\n        galat.append("Akhir data uji harus setelah akhir data latih.")\n    spek = replace(peng.spek, exog=tuple(f"x{i}" for i in range(len(peng.kolom_x) * len(set(peng.lag_x)))))\n    n_latih = int(seri.loc[peng.awal_latih:peng.akhir_latih].notna().sum())\n    minimal = spek.minimal_observasi()\n    if n_latih < minimal:\n        galat.append(f"Data latih hanya {n_latih} observasi; model {peng.spek.label()} butuh minimal ±{minimal}. "\n                     "Perpanjang data latih atau sederhanakan model.")\n    P, D, Q, s = peng.spek.seasonal_order\n    if (P or D or Q) and s < 2:\n        galat.append("Komponen musiman (P, D, Q) membutuhkan panjang musim s ≥ 2. "\n                     "Untuk data tanpa musim, isi P = D = Q = 0.")\n    if peng.horizon < 1:\n        galat.append("Horizon proyeksi minimal 1 periode.")\n    if any(lag < 0 for lag in peng.lag_x):\n        galat.append("Lag X tidak boleh negatif.")\n    galat += siapkan_desain_x(X_dasar, peng)[2]\n    return galat\n\n\ndef _label(spek: am.SpesifikasiModel, varian: str = "") -> str:\n    return spek.label(dengan_x=False) + (f" — {varian}" if varian else "")\n\n\ndef jalankan(seri: pd.Series, peng: Pengaturan, X_dasar: pd.DataFrame | None = None) -> HasilProyeksi:\n    """Jalankan seluruh alur. X_dasar: tabel X mentah (indeks periode, boleh memuat periode masa depan)."""\n    galat = periksa(seri, peng, X_dasar)\n    if galat:\n        raise GalatPengaturan("\\n".join(galat))\n\n    alpha = 1 - peng.tingkat_kepercayaan / 100\n    y_model = ke_skala_model(seri, peng.transformasi)\n    y_asli = ke_skala_asli(y_model, peng.transformasi)\n    X_model, X_desain, _ = siapkan_desain_x(X_dasar, peng)\n    spek = replace(peng.spek, exog=tuple(X_desain.columns) if X_desain is not None else ())\n    spek_0 = spek.tanpa_x()\n    s = spek.seasonal_order[3]\n    pakai_x = bool(spek.exog)\n    peringatan, info_x = [], {}\n    if pakai_x:\n        info_x["Kolom X dalam model"] = ", ".join(spek.exog)\n\n    # ------------------------------------------------------------------ backtest\n    bt = bt0 = akurasi = None\n    if peng.akhir_uji is not None:\n        y_uji = y_model.loc[peng.akhir_latih + 1:peng.akhir_uji]\n        lain = {}\n        if pakai_x and peng.proyeksi_x_sarima:\n            Xf, info = fc.ramal_x_sarima(X_model, peng.akhir_latih, len(y_uji), s, peng.maks_orde_x)\n            lain[NAMA_EX_ANTE] = fc.desain_x_depan(X_model, Xf, peng.lag_x)\n            info_x.update({f"Proyeksi X (backtest) — {k}": v for k, v in info.items()})\n        bt = fc.backtest(y_model, spek, peng.awal_latih, peng.akhir_latih, peng.akhir_uji,\n                         peng.transformasi, alpha, peng.koreksi_bias, X=X_desain, X_uji_lain=lain)\n        aktual = bt["tabel"]["aktual"]\n        if pakai_x:\n            ramal = {_label(spek, NAMA_EX_POST_UJI): bt["tabel"]["ramalan"]}\n            ramal.update({_label(spek, k): t["ramalan"] for k, t in bt["tabel_varian"].items()})\n            bt0 = fc.backtest(y_model, spek_0, bt["y_latih"].index[0], peng.akhir_latih, peng.akhir_uji,\n                              peng.transformasi, alpha, peng.koreksi_bias)\n            ramal[_label(spek_0, NAMA_TANPA_X)] = bt0["tabel"]["ramalan"]\n        else:\n            ramal = {spek.label(): bt["tabel"]["ramalan"]}\n        akurasi = ev.tabel_akurasi(aktual, {**ramal, **bt["pembanding"]})\n        if aktual.isna().any():\n            peringatan.append(f"{int(aktual.isna().sum())} periode uji tanpa nilai aktual tidak dihitung di metrik.")\n\n    # ------------------------------------------------------------------ proyeksi ke depan\n    akhir_data = y_model.loc[peng.awal_latih:].last_valid_index()\n    y_final = y_model.loc[peng.awal_latih:akhir_data]\n    hasil_akhir = am.estimasi(y_final, spek, X_desain)\n    index_depan = pd.period_range(akhir_data + 1, periods=peng.horizon, freq=y_model.index.freq)\n    ramalan_varian, hasil_0, nama_utama = {}, None, ""\n    if pakai_x:\n        X_manual = X_desain.reindex(index_depan)\n        lengkap = X_manual.notna().all(axis=1).to_numpy()\n        n_manual = int(lengkap.argmin()) if not lengkap.all() else len(lengkap)\n        if n_manual:\n            ramalan_varian[NAMA_EX_POST_DEPAN] = fc.proyeksi(hasil_akhir, n_manual, peng.transformasi, alpha,\n                                                             peng.koreksi_bias, X_manual.iloc[:n_manual])\n            if n_manual < peng.horizon:\n                peringatan.append(f"Nilai X manual hanya tersedia {n_manual} dari {peng.horizon} periode ke depan; "\n                                  "ramalan ex-post dibatasi sepanjang itu.")\n        if peng.proyeksi_x_sarima:\n            Xf, info = fc.ramal_x_sarima(X_model, akhir_data, peng.horizon, s, peng.maks_orde_x)\n            info_x.update({f"Proyeksi X (ke depan) — {k}": v for k, v in info.items()})\n            ramalan_varian[NAMA_EX_ANTE] = fc.proyeksi(hasil_akhir, peng.horizon, peng.transformasi, alpha,\n                                                       peng.koreksi_bias, fc.desain_x_depan(X_model, Xf, peng.lag_x))\n        if not ramalan_varian:\n            raise GalatPengaturan("Nilai X untuk periode ke depan tidak ada. Isi nilai X pada baris tanggal ke "\n                                  "depan di file data, atau aktifkan proyeksi X dengan SARIMA.")\n        hasil_0 = am.estimasi(y_final.loc[hasil_akhir.y_model.index[0]:], spek_0)\n        ramalan_varian[NAMA_TANPA_X] = fc.proyeksi(hasil_0, peng.horizon, peng.transformasi, alpha,\n                                                   peng.koreksi_bias)\n        if NAMA_EX_POST_DEPAN in ramalan_varian and n_manual == peng.horizon:\n            nama_utama = NAMA_EX_POST_DEPAN\n        else:\n            nama_utama = NAMA_EX_ANTE if NAMA_EX_ANTE in ramalan_varian else NAMA_EX_POST_DEPAN\n        ramalan = ramalan_varian[nama_utama]\n    else:\n        ramalan = fc.proyeksi(hasil_akhir, peng.horizon, peng.transformasi, alpha, peng.koreksi_bias)\n\n    if peng.akhir_uji is not None and akhir_data > peng.akhir_latih:\n        peringatan.append(f"Proyeksi ke depan memakai model yang diestimasi ulang dengan data "\n                          f"{label_periode(hasil_akhir.y_model.index[0])} s.d. {label_periode(akhir_data)} "\n                          "(termasuk periode uji), sehingga koefisiennya bisa berbeda dari model backtest.")\n\n    return HasilProyeksi(\n        pengaturan=peng, y_asli=y_asli, backtest=bt, akurasi=akurasi,\n        hasil_akhir=hasil_akhir, koefisien_akhir=am.tabel_koefisien(hasil_akhir),\n        statistik_akhir=am.statistik_ringkas(hasil_akhir), diagnostik_akhir=am.diagnostik(hasil_akhir),\n        ramalan=ramalan, akhir_data_proyeksi=akhir_data, peringatan=peringatan,\n        ramalan_varian=ramalan_varian, nama_ramalan_utama=nama_utama, hasil_tanpa_x=hasil_0,\n        backtest_tanpa_x=bt0, info_x=info_x, X_model=X_model,\n    )\n\n\n# --------------------------------------------------------------------------- tabel ringkasan\n\ndef tabel_hasil(h: HasilProyeksi) -> pd.DataFrame:\n    """Tabel gabungan: periode uji (aktual, ramalan, galat) + proyeksi ke depan (ramalan utama)."""\n    bagian = []\n    if h.backtest is not None:\n        t = h.backtest["tabel"].copy()\n        t.insert(0, "Jenis", "Uji (backtest)" + (f" — {NAMA_EX_POST_UJI}" if h.pakai_x else ""))\n        bagian.append(t)\n    r = h.ramalan[["ramalan", "batas_bawah", "batas_atas"]].copy()\n    r.insert(0, "Jenis", "Proyeksi ke depan" + (f" — {h.nama_ramalan_utama}" if h.pakai_x else ""))\n    r["aktual"], r["galat"] = float("nan"), float("nan")\n    bagian.append(r)\n    tabel = pd.concat(bagian)[["Jenis", "aktual", "ramalan", "galat", "batas_bawah", "batas_atas"]]\n    tabel.index = [label_periode(p) for p in tabel.index]\n    tabel.index.name = "Periode"\n    return tabel.rename(columns={"aktual": "Aktual", "ramalan": "Ramalan", "galat": "Galat (aktual − ramalan)",\n                                 "batas_bawah": "Batas bawah", "batas_atas": "Batas atas"})\n\n\ndef tabel_perbandingan_ramalan(h: HasilProyeksi) -> pd.DataFrame:\n    """Ramalan semua varian berdampingan (kolom), untuk periode uji dan ke depan. Kosong bila tanpa X."""\n    if not h.pakai_x:\n        return pd.DataFrame()\n    bagian = []\n    if h.backtest is not None:\n        kol = {"Aktual": h.backtest["tabel"]["aktual"], NAMA_EX_POST_UJI: h.backtest["tabel"]["ramalan"]}\n        kol.update({k: t["ramalan"] for k, t in h.backtest["tabel_varian"].items()})\n        kol[NAMA_TANPA_X] = h.backtest_tanpa_x["tabel"]["ramalan"]\n        t = pd.DataFrame(kol)\n        t.insert(0, "Jenis", "Uji (backtest)")\n        bagian.append(t)\n    kol = {k: r["ramalan"] for k, r in h.ramalan_varian.items()}\n    t = pd.DataFrame(kol)\n    t.insert(0, "Jenis", "Proyeksi ke depan")\n    bagian.append(t)\n    tabel = pd.concat(bagian)\n    tabel.index = [label_periode(p) for p in tabel.index]\n    tabel.index.name = "Periode"\n    return tabel\n\n\ndef tabel_perbandingan_model(h: HasilProyeksi, pakai_backtest: bool = True) -> pd.DataFrame:\n    """Statistik model dengan X vs tanpa X (sampel sama). Kosong bila tanpa X."""\n    if not h.pakai_x:\n        return pd.DataFrame()\n    if pakai_backtest and h.backtest is not None:\n        m_x, m_0 = h.backtest["hasil"], h.backtest_tanpa_x["hasil"]\n    else:\n        m_x, m_0 = h.hasil_akhir, h.hasil_tanpa_x\n    pilih = ["Jumlah observasi", "Log likelihood", "Akaike info criterion (AIC)", "Schwarz criterion (SC/BIC)",\n             "Hannan-Quinn criterion (HQ)", "R-squared", "Adjusted R-squared", "S.E. of regression",\n             "Durbin-Watson stat"]\n    return pd.DataFrame({\n        f"Dengan X — {m_x.spek.label()}": am.statistik_ringkas(m_x)[pilih],\n        f"Tanpa X — {m_0.spek.label()}": am.statistik_ringkas(m_0)[pilih],\n    })\n',
    'modules/visualization.py': '"""Grafik: Plotly untuk layar (interaktif), Matplotlib untuk file PNG unduhan.\n\nPlotly diimpor di dalam fungsi agar fungsi PNG bisa diuji tanpa Plotly."""\nfrom __future__ import annotations\n\nimport io\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt  # noqa: E402\nimport pandas as pd  # noqa: E402\n\n# Grafik memakai warna kontras agar mudah dibedakan (tema aplikasi navy-biru-putih-abu).\nWARNA = {\n    "latih": "#0B1F3A",            # navy: data aktual\n    "uji": "#16A34A",              # hijau: aktual periode uji\n    "backtest": "#DC2626",         # merah: ramalan periode uji\n    "proyeksi": "#DC2626",         # merah: proyeksi ke depan\n    "pita_backtest": "rgba(220,38,38,0.12)",\n    "pita_proyeksi": "rgba(220,38,38,0.12)",\n    "acf": "#1D4ED8",\n}\nWARNA_VARIAN = ["#F59E0B", "#7C3AED", "#0891B2", "#DB2777"]   # oranye, ungu, sian, merah muda\nWARNA_BATANG = ["#DC2626", "#F59E0B", "#7C3AED", "#16A34A", "#64748B", "#94A3B8", "#CBD5E1"]\nTATA_LETAK = dict(template="plotly_white", height=460, margin=dict(t=50, b=10, l=10, r=10),\n                  legend=dict(orientation="h", y=-0.18), hovermode="x unified",\n                  font=dict(color="#0B1F3A"), title_font=dict(color="#0B1F3A", size=16))\n\n\ndef _x(index: pd.PeriodIndex):\n    return index.to_timestamp()\n\n\ndef grafik_seri(seri: pd.Series, judul: str = "", label_y: str = "Nilai"):\n    import plotly.graph_objects as go\n    fig = go.Figure(go.Scatter(x=_x(seri.index), y=seri, mode="lines+markers", name=seri.name or "Data",\n                               line=dict(color=WARNA["latih"]), marker=dict(size=4)))\n    fig.update_layout(title=judul, yaxis_title=label_y, **TATA_LETAK)\n    return fig\n\n\ndef _pita(fig, index, bawah, atas, warna, nama):\n    fig.add_scatter(x=_x(index), y=atas, line=dict(width=0), showlegend=False, hoverinfo="skip")\n    fig.add_scatter(x=_x(index), y=bawah, fill="tonexty", fillcolor=warna, line=dict(width=0), name=nama,\n                    hoverinfo="skip")\n\n\ndef _label_model(h) -> str:\n    spek = getattr(h, "spek", None) or h.pengaturan.spek\n    return spek.label()\n\n\ndef _garis_varian(h) -> list:\n    """Garis tambahan untuk SARIMAX: (index, nilai, nama) varian ramalan selain ramalan utama."""\n    garis = []\n    if not getattr(h, "pakai_x", False):\n        return garis\n    if h.backtest is not None:\n        for nama, t in h.backtest["tabel_varian"].items():\n            garis.append((t.index, t["ramalan"], f"Uji — {nama}"))\n        t0 = h.backtest_tanpa_x["tabel"]\n        garis.append((t0.index, t0["ramalan"], "Uji — tanpa X"))\n    for nama, r in h.ramalan_varian.items():\n        if nama != h.nama_ramalan_utama:\n            garis.append((r.index, r["ramalan"], f"Proyeksi — {nama}"))\n    return garis\n\n\ndef grafik_hasil(h, label_y: str = "Nilai"):\n    import plotly.graph_objects as go\n    """Data latih, data uji, ramalan backtest, proyeksi ke depan, dan selang kepercayaan."""\n    peng = h.pengaturan\n    tingkat = peng.tingkat_kepercayaan\n    fig = go.Figure()\n    latih = h.y_asli.loc[peng.awal_latih:peng.akhir_latih]\n    fig.add_scatter(x=_x(latih.index), y=latih, name="Aktual — data latih", mode="lines",\n                    line=dict(color=WARNA["latih"]))\n    if h.backtest is not None:\n        t = h.backtest["tabel"]\n        _pita(fig, t.index, t["batas_bawah"], t["batas_atas"], WARNA["pita_backtest"], f"Selang {tingkat}% (uji)")\n        fig.add_scatter(x=_x(t.index), y=t["aktual"], name="Aktual — data uji", mode="lines+markers",\n                        line=dict(color=WARNA["uji"]))\n        fig.add_scatter(x=_x(t.index), y=t["ramalan"],\n                        name="Ramalan periode uji" + (" — ex-post (X aktual)" if getattr(h, "pakai_x", False) else ""),\n                        mode="lines+markers", line=dict(color=WARNA["backtest"], dash="dash"))\n    sesudah = h.y_asli.loc[(peng.akhir_uji or peng.akhir_latih) + 1:h.akhir_data_proyeksi]\n    if len(sesudah.dropna()):\n        fig.add_scatter(x=_x(sesudah.index), y=sesudah, name="Aktual — setelah periode uji", mode="lines",\n                        line=dict(color=WARNA["latih"], dash="dot"))\n    r = h.ramalan\n    _pita(fig, r.index, r["batas_bawah"], r["batas_atas"], WARNA["pita_proyeksi"], f"Selang {tingkat}% (proyeksi)")\n    utama = getattr(h, "nama_ramalan_utama", "")\n    fig.add_scatter(x=_x(r.index), y=r["ramalan"], name="Proyeksi ke depan" + (f" — {utama}" if utama else ""),\n                    mode="lines+markers", line=dict(color=WARNA["proyeksi"]))\n    for i, (idx, nilai, nama) in enumerate(_garis_varian(h)):\n        fig.add_scatter(x=_x(idx), y=nilai, name=nama, mode="lines+markers",\n                        line=dict(color=WARNA_VARIAN[i % len(WARNA_VARIAN)], dash="dot"), marker=dict(size=4))\n    fig.update_layout(title=f"{_label_model(h)} — aktual dan ramalan", yaxis_title=label_y, **TATA_LETAK)\n    return fig\n\n\ndef grafik_acf(diag: dict):\n    import plotly.graph_objects as go\n    a = diag["acf"]\n    fig = go.Figure(go.Bar(x=a["lag"], y=a["acf"], marker_color=WARNA["acf"], name="ACF"))\n    for tanda in (1, -1):\n        fig.add_hline(y=tanda * diag["batas_acf"], line_dash="dash", line_color="#DC2626")\n    fig.update_layout(title="ACF residual (garis putus = batas signifikansi 95%)", xaxis_title="Lag",\n                      **{**TATA_LETAK, "height": 320, "hovermode": "closest"})\n    return fig\n\n\ndef grafik_akurasi(akurasi: pd.DataFrame, ukuran: str = "MAPE (%)"):\n    """Batang horizontal satu ukuran akurasi untuk semua model/pembanding (makin pendek makin baik)."""\n    import plotly.graph_objects as go\n    data = akurasi[ukuran].dropna().sort_values(ascending=False)\n    warna = [WARNA_BATANG[list(akurasi.index).index(i) % len(WARNA_BATANG)] for i in data.index]\n    fig = go.Figure(go.Bar(x=data.values, y=list(data.index), orientation="h", marker_color=warna,\n                           text=[f"{v:,.2f}" for v in data.values], textposition="outside"))\n    fig.update_layout(title=f"{ukuran} pada periode uji (makin kecil makin baik)",\n                      **{**TATA_LETAK, "height": 120 + 45 * len(data), "hovermode": "closest",\n                         "margin": dict(t=50, b=10, l=10, r=60)})\n    return fig\n\n\ndef grafik_x(X, judul: str = "Variabel X"):\n    """Satu garis per variabel X (sumbu-y sendiri-sendiri lewat subplot)."""\n    import plotly.graph_objects as go\n    from plotly.subplots import make_subplots\n    fig = make_subplots(rows=len(X.columns), cols=1, shared_xaxes=True, subplot_titles=list(X.columns))\n    for i, k in enumerate(X.columns, start=1):\n        fig.add_trace(go.Scatter(x=_x(X.index), y=X[k], mode="lines+markers", name=k, showlegend=False,\n                                 line=dict(color=WARNA_VARIAN[(i - 1) % len(WARNA_VARIAN)]), marker=dict(size=3)),\n                      row=i, col=1)\n    fig.update_layout(title=judul, **{**TATA_LETAK, "height": 120 + 170 * len(X.columns), "hovermode": "closest"})\n    return fig\n\n\ndef png_hasil(h, label_y: str = "Nilai") -> bytes:\n    """Versi Matplotlib dari grafik_hasil, untuk diunduh sebagai PNG."""\n    peng = h.pengaturan\n    fig, ax = plt.subplots(figsize=(11, 5), dpi=150)\n    latih = h.y_asli.loc[peng.awal_latih:peng.akhir_latih]\n    ax.plot(_x(latih.index), latih, color=WARNA["latih"], label="Aktual — data latih")\n    if h.backtest is not None:\n        t = h.backtest["tabel"]\n        ax.fill_between(_x(t.index), t["batas_bawah"], t["batas_atas"], color=WARNA["backtest"], alpha=0.15)\n        ax.plot(_x(t.index), t["aktual"], color=WARNA["uji"], marker="o", ms=3, label="Aktual — data uji")\n        ax.plot(_x(t.index), t["ramalan"], color=WARNA["backtest"], ls="--", marker="o", ms=3,\n                label="Ramalan periode uji" + (" — ex-post (X aktual)" if getattr(h, "pakai_x", False) else ""))\n    sesudah = h.y_asli.loc[(peng.akhir_uji or peng.akhir_latih) + 1:h.akhir_data_proyeksi]\n    if len(sesudah.dropna()):\n        ax.plot(_x(sesudah.index), sesudah, color=WARNA["latih"], ls=":", label="Aktual — setelah periode uji")\n    r = h.ramalan\n    ax.fill_between(_x(r.index), r["batas_bawah"], r["batas_atas"], color=WARNA["proyeksi"], alpha=0.15,\n                    label=f"Selang {peng.tingkat_kepercayaan}%")\n    utama = getattr(h, "nama_ramalan_utama", "")\n    ax.plot(_x(r.index), r["ramalan"], color=WARNA["proyeksi"], marker="o", ms=3,\n            label="Proyeksi ke depan" + (f" — {utama}" if utama else ""))\n    for i, (idx, nilai, nama) in enumerate(_garis_varian(h)):\n        ax.plot(_x(idx), nilai, color=WARNA_VARIAN[i % len(WARNA_VARIAN)], ls=":", marker="o", ms=2, label=nama)\n    ax.set_title(f"{_label_model(h)} — aktual dan ramalan")\n    ax.set_ylabel(label_y)\n    ax.grid(alpha=0.3)\n    ax.legend(loc="upper left", fontsize=8)\n    ax.ticklabel_format(axis="y", style="plain")\n    fig.autofmt_xdate()\n    buf = io.BytesIO()\n    fig.savefig(buf, format="png", bbox_inches="tight")\n    plt.close(fig)\n    return buf.getvalue()\n',
    'modules/export_results.py': '"""Membuat file unduhan dari hasil model yang benar-benar dijalankan."""\nfrom __future__ import annotations\n\nimport io\nfrom datetime import datetime\n\nimport pandas as pd\n\nfrom .data_preprocessing import TRANSFORMASI, label_periode\nfrom .pipeline import HasilProyeksi, tabel_hasil, tabel_perbandingan_model, tabel_perbandingan_ramalan\n\n\ndef tabel_pengaturan(h: HasilProyeksi) -> pd.DataFrame:\n    p = h.pengaturan\n    baris = {\n        "Model": h.spek.label(),\n        "Order (p,d,q)": str(p.spek.order),\n        "Seasonal order (P,D,Q,s)": str(p.spek.seasonal_order),\n        "Konstanta C": "Ya" if p.spek.konstanta else "Tidak",\n        "Transformasi": TRANSFORMASI[p.transformasi],\n        "Variabel X": ", ".join(p.kolom_x) if p.kolom_x else "-",\n        "Transformasi X / lag X": f"{p.transformasi_x} / {\', \'.join(map(str, p.lag_x))}" if p.kolom_x else "-",\n        "Data latih": f"{label_periode(p.awal_latih)} s.d. {label_periode(p.akhir_latih)}",\n        "Data uji": f"{label_periode(p.akhir_latih + 1)} s.d. {label_periode(p.akhir_uji)}" if p.akhir_uji else "-",\n        "Data untuk proyeksi": f"{label_periode(p.awal_latih)} s.d. {label_periode(h.akhir_data_proyeksi)}",\n        "Horizon": p.horizon,\n        "Tingkat kepercayaan": f"{p.tingkat_kepercayaan}%",\n        "Koreksi bias exp()": "Ya" if p.koreksi_bias else "Tidak",\n        "Paksa stasioner / invertible": f"{p.spek.paksa_stasioner} / {p.spek.paksa_invertible}",\n        "Dibuat": datetime.now().strftime("%Y-%m-%d %H:%M"),\n    }\n    baris.update(h.info_x)\n    return pd.DataFrame({"Nilai": baris}).rename_axis("Pengaturan")\n\n\ndef csv_hasil(h: HasilProyeksi) -> bytes:\n    return tabel_hasil(h).to_csv().encode("utf-8-sig")  # utf-8-sig agar terbaca benar di Excel\n\n\ndef excel_laporan(h: HasilProyeksi) -> bytes:\n    buf = io.BytesIO()\n    with pd.ExcelWriter(buf, engine="openpyxl") as w:\n        tabel_pengaturan(h).to_excel(w, sheet_name="Pengaturan")\n        tabel_hasil(h).to_excel(w, sheet_name="Hasil")\n        if h.akurasi is not None:\n            h.akurasi.to_excel(w, sheet_name="Evaluasi")\n        h.koefisien_akhir.to_excel(w, sheet_name="Koefisien")\n        h.statistik_akhir.to_frame("Nilai").to_excel(w, sheet_name="Statistik model")\n        if h.backtest is not None:\n            from .arima_model import tabel_koefisien\n            tabel_koefisien(h.backtest["hasil"]).to_excel(w, sheet_name="Koefisien backtest")\n        if h.pakai_x:\n            from .arima_model import statistik_ringkas, tabel_koefisien\n            tabel_perbandingan_ramalan(h).to_excel(w, sheet_name="Perbandingan ramalan")\n            tabel_perbandingan_model(h, pakai_backtest=h.backtest is not None).to_excel(\n                w, sheet_name="Perbandingan model")\n            tabel_koefisien(h.hasil_tanpa_x).to_excel(w, sheet_name="Koefisien tanpa X")\n            statistik_ringkas(h.hasil_tanpa_x).to_frame("Nilai").to_excel(w, sheet_name="Statistik tanpa X")\n    return buf.getvalue()\n\n\ndef excel_hasil_saja(h: HasilProyeksi) -> bytes:\n    buf = io.BytesIO()\n    with pd.ExcelWriter(buf, engine="openpyxl") as w:\n        tabel_hasil(h).to_excel(w, sheet_name="Hasil")\n    return buf.getvalue()\n\n\ndef teks_ringkasan(h: HasilProyeksi) -> bytes:\n    from .arima_model import ringkasan_teks\n    bagian = ["ARIMA Forecasting Platform — ringkasan model", "=" * 60, "",\n              tabel_pengaturan(h).to_string(), "",\n              "MODEL UNTUK PROYEKSI KE DEPAN", "-" * 60,\n              ringkasan_teks(h.hasil_akhir, h.statistik_akhir)]\n    if h.pakai_x:\n        from .arima_model import statistik_ringkas\n        bagian += ["", "MODEL PEMBANDING TANPA X (sampel sama)", "-" * 60,\n                   ringkasan_teks(h.hasil_tanpa_x, statistik_ringkas(h.hasil_tanpa_x)), "",\n                   "PERBANDINGAN RAMALAN", "-" * 60,\n                   tabel_perbandingan_ramalan(h).to_string(float_format=lambda v: f"{v:,.2f}")]\n    if h.diagnostik_akhir["peringatan"]:\n        bagian += ["", "PERINGATAN MODEL"] + [f"- {p}" for p in h.diagnostik_akhir["peringatan"]]\n    if h.akurasi is not None:\n        bagian += ["", "EVALUASI PERIODE UJI", "-" * 60, h.akurasi.to_string(float_format=lambda v: f"{v:,.4f}")]\n    if h.peringatan:\n        bagian += ["", "CATATAN"] + [f"- {p}" for p in h.peringatan]\n    return "\\n".join(bagian).encode("utf-8")\n',
}

for nama, isi in BERKAS_MESIN.items():
    path = FOLDER / nama
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(isi)

# Data contoh SINTETIS (bukan data asli) — hanya untuk mencoba notebook.
# Kolom: tanggal, nilai (target), dummy_libur (0/1), indeks_harga; 6 baris terakhir = periode ke depan (X terisi, nilai kosong).
import numpy as np, pandas as pd
rng = np.random.default_rng(7)
_per = pd.period_range("2019-01", periods=78, freq="M")
_t = np.arange(len(_per))
_dummy = (_per.month.to_numpy() == 12).astype(int)
_indeks = np.round(100 * np.exp(0.003 * _t + rng.normal(0, 0.02, len(_per))), 2)
_pola = 0.5 * np.sin(2 * np.pi * (_per.month.to_numpy() - 3) / 12)
_logy = 10 + 0.4 * _dummy + 0.8 * np.log(_indeks / 100) + _pola + rng.normal(0, 0.12, len(_per))
_nilai = np.round(np.exp(_logy))
_nilai[72:] = np.nan
_contoh = pd.DataFrame({"tanggal": _per.to_timestamp(how="end").normalize().date, "nilai": _nilai,
                        "dummy_libur": _dummy, "indeks_harga": _indeks})
(FOLDER / "sample_data").mkdir(exist_ok=True)
_contoh.to_excel(FOLDER / "sample_data" / "sample_dataset.xlsx", index=False)
_contoh.to_csv(FOLDER / "sample_data" / "sample_dataset.csv", index=False)

import modules.data_loader as dl, modules.data_preprocessing as dp, modules.exploration as ex
import modules.arima_model as am, modules.forecasting as fc, modules.evaluation as ev
import modules.pipeline as pl, modules.visualization as vz, modules.export_results as er
for _m in (dl, dp, ex, am, fc, ev, pl, vz, er):
    importlib.reload(_m)

import statsmodels, plotly
from IPython.display import display, Markdown
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print(f"✅ Mesin siap. statsmodels {statsmodels.__version__}, pandas {pd.__version__}, plotly {plotly.__version__}")

In [ ]:
#@title Langkah 2 — Unggah data
#@markdown Centang untuk mencoba dengan data contoh sintetis. Hapus centang untuk mengunggah file Anda (.xlsx / .csv).
PAKAI_DATA_CONTOH = False #@param {type:"boolean"}

if PAKAI_DATA_CONTOH:
    NAMA_FILE = str(FOLDER / "sample_data" / "sample_dataset.xlsx")
    print("Memakai data contoh SINTETIS.")
else:
    from google.colab import files
    print("Klik 'Choose Files' lalu pilih file data Anda...")
    _up = files.upload()
    if not _up:
        raise SystemExit("Tidak ada file yang diunggah.")
    NAMA_FILE = str(FOLDER / list(_up)[0])

try:
    df_mentah = dl.baca_file(NAMA_FILE)
except dl.GalatData as e:
    raise SystemExit(f"❌ {e}")
ringkas = dl.ringkasan_dataset(df_mentah)
print(f"✅ {Path(NAMA_FILE).name}: {ringkas['jumlah_baris']} baris, {ringkas['jumlah_kolom']} kolom")
display(ringkas["info_kolom"])
display(df_mentah.head(10))

In [ ]:
#@title Langkah 3 — Pilih kolom & periksa data
#@markdown Tulis nama kolom persis seperti di file. **Kosongkan** agar dipilih otomatis. Frekuensi `Otomatis` ditebak dari kolom tanggal.
KOLOM_TANGGAL = "" #@param {type:"string"}
KOLOM_TARGET = "" #@param {type:"string"}
FREKUENSI = "Otomatis" #@param ["Otomatis", "Bulanan", "Triwulanan", "Tahunan", "Mingguan", "Harian"]
#@markdown **Variabel independen (X)**: `otomatis` = semua kolom angka selain tanggal & target; `tanpa` = SARIMA biasa; atau tulis nama kolom dipisah koma.
KOLOM_X = "otomatis" #@param {type:"string"}

_tgl, _tgt = dl.tebak_kolom(df_mentah)
KOLOM_TANGGAL = KOLOM_TANGGAL.strip() or _tgl
KOLOM_TARGET = KOLOM_TARGET.strip() or _tgt
for _k in (KOLOM_TANGGAL, KOLOM_TARGET):
    if _k not in df_mentah.columns:
        raise SystemExit(f"❌ Kolom '{_k}' tidak ada. Kolom yang tersedia: {list(df_mentah.columns)}")
if FREKUENSI == "Otomatis":
    FREKUENSI = dp.tebak_frekuensi(df_mentah[KOLOM_TANGGAL])
    print(f"Frekuensi terdeteksi: {FREKUENSI} (ubah di formulir bila keliru)")

_kx = KOLOM_X.strip()
if _kx.lower() in ("", "tanpa", "-"):
    kolom_x = []
elif _kx.lower() == "otomatis":
    kolom_x = dp.kolom_x_otomatis(df_mentah, KOLOM_TANGGAL, KOLOM_TARGET)
else:
    kolom_x = [k.strip() for k in _kx.split(",") if k.strip()]
    _salah = [k for k in kolom_x if k not in df_mentah.columns or k in (KOLOM_TANGGAL, KOLOM_TARGET)]
    if _salah:
        raise SystemExit(f"❌ Kolom X tidak valid: {_salah}. Kolom yang tersedia: {list(df_mentah.columns)}")
print(f"Kolom tanggal: {KOLOM_TANGGAL} | Kolom target: {KOLOM_TARGET} | Frekuensi: {FREKUENSI}")
print(f"Variabel X: {', '.join(kolom_x) if kolom_x else '— (SARIMA tanpa X)'}")

validasi = dp.siapkan_seri(df_mentah, KOLOM_TANGGAL, KOLOM_TARGET, FREKUENSI)
for _p in validasi.peringatan:
    print("⚠️", _p)
if not validasi.ok:
    raise SystemExit("❌ " + "\n❌ ".join(validasi.galat))
seri = validasi.seri
KODE_FREK, S_BAWAAN = dp.FREKUENSI[FREKUENSI]
_akhir_y = seri.last_valid_index()
print(f"✅ Data siap: target terisi {dp.label_periode(seri.first_valid_index())} – {dp.label_periode(_akhir_y)}, "
      f"{int(seri.loc[:_akhir_y].isna().sum())} kosong")

X_dasar = None
if kolom_x:
    _vx = dp.siapkan_x(df_mentah, KOLOM_TANGGAL, kolom_x, FREKUENSI, seri.index)
    for _p in _vx.peringatan:
        print("⚠️", _p)
    if not _vx.ok:
        raise SystemExit("❌ " + "\n❌ ".join(_vx.galat))
    X_dasar = _vx.X
    _ringkas_x = pd.DataFrame({
        "Jenis": ["dummy (0/1)" if dp.adalah_dummy(X_dasar[k]) else "kontinu" for k in kolom_x],
        "Terisi s.d. akhir target": [int(X_dasar[k].loc[:_akhir_y].notna().sum()) for k in kolom_x],
        "Kosong s.d. akhir target": [int(X_dasar[k].loc[:_akhir_y].isna().sum()) for k in kolom_x],
        "Nilai manual ke depan": [int(X_dasar[k].loc[_akhir_y + 1:].notna().sum()) for k in kolom_x],
    }, index=kolom_x)
    display(_ringkas_x)

vz.grafik_seri(seri.loc[:_akhir_y], f"{KOLOM_TARGET} ({FREKUENSI.lower()})", KOLOM_TARGET).show()
display(ex.statistik_deskriptif(seri.loc[:_akhir_y]).to_frame("Nilai"))


In [ ]:
#@title Langkah 4 — Pengaturan model
#@markdown **Transformasi Y**: `log` = data di-log dulu (data asli > 0); `sudah_log` = kolom sudah berupa log; `tanpa` = apa adanya.
TRANSFORMASI = "log" #@param ["log", "sudah_log", "tanpa"]
#@markdown **Periode** (format bulanan `2021-01`, triwulanan `2021Q1`, tahunan `2021`, mingguan/harian `2021-01-04`). **Kosongkan untuk otomatis**: data uji = satu musim terakhir (atau ±20% data). Centang TANPA_UJI bila tidak ingin data uji.
AWAL_LATIH = "" #@param {type:"string"}
AKHIR_LATIH = "" #@param {type:"string"}
AKHIR_UJI = "" #@param {type:"string"}
TANPA_UJI = False #@param {type:"boolean"}
#@markdown **Orde model** SARIMA(p,d,q)(P,D,Q)s. `s = 0` → otomatis dari frekuensi (12 bulanan, 4 triwulanan, 52 mingguan, 7 harian, 1 tahunan). Untuk data tanpa musim (s = 1), P, D, Q diabaikan.
p = 0 #@param {type:"integer"}
d = 0 #@param {type:"integer"}
q = 0 #@param {type:"integer"}
P = 1 #@param {type:"integer"}
D = 0 #@param {type:"integer"}
Q = 0 #@param {type:"integer"}
s = 0 #@param {type:"integer"}
KONSTANTA = True #@param {type:"boolean"}
#@markdown **Pilih orde otomatis** (opsional): estimasi semua kombinasi ARMA (p,q)(P,Q) hingga batas maksimum pada **data latih** (d, D, X, dan konstanta mengikuti pengaturan di atas), lalu pakai yang kriterianya terkecil. Isian p, q, P, Q di atas diabaikan.
PILIH_ORDE_OTOMATIS = False #@param {type:"boolean"}
MAKS_p = 2 #@param {type:"integer"}
MAKS_q = 2 #@param {type:"integer"}
MAKS_P = 1 #@param {type:"integer"}
MAKS_Q = 1 #@param {type:"integer"}
KRITERIA = "AIC" #@param ["AIC", "SC", "HQ"]
#@markdown **Variabel X** (hanya bila Langkah 3 memilih kolom X). `log` tidak berlaku untuk kolom dummy 0/1. **LAG_X**: mis. `0` (X periode yang sama) atau `0,1` (tambah X periode sebelumnya).
TRANSFORMASI_X = "tanpa" #@param ["tanpa", "log"]
LAG_X = "0" #@param {type:"string"}
PROYEKSI_X_SARIMA = True #@param {type:"boolean"}
#@markdown **Proyeksi** — `HORIZON = 0` → otomatis (satu musim, atau 5 periode untuk data tanpa musim).
HORIZON = 0 #@param {type:"integer"}
TINGKAT_KEPERCAYAAN = 95 #@param [80, 90, 95, 99] {type:"raw"}
#@markdown **Opsi lanjutan** (bawaan: mati)
PAKSA_STASIONER = False #@param {type:"boolean"}
PAKSA_INVERTIBLE = False #@param {type:"boolean"}
KOREKSI_BIAS = False #@param {type:"boolean"}

from dataclasses import replace as _ganti

S_MODEL = int(s) if int(s) > 0 else S_BAWAAN
if S_MODEL < 2 and (P or D or Q):
    print(f"ℹ️ Data tanpa musim (s = {S_MODEL}): komponen musiman P, D, Q diabaikan.")
    P = D = Q = 0
HORIZON_MODEL = int(HORIZON) if int(HORIZON) > 0 else (S_MODEL if S_MODEL > 1 else 5)
try:
    LAG_MODEL = tuple(sorted({int(v) for v in str(LAG_X).replace(" ", "").split(",") if v != ""})) or (0,)
except ValueError:
    raise SystemExit("❌ LAG_X harus berupa angka dipisah koma, mis. 0 atau 0,1")

def _periode(teks, cadangan):
    teks = str(teks).strip()
    return pd.Period(teks, freq=KODE_FREK) if teks else cadangan

_awal, _akhir_latih, _akhir_uji = pl.periode_otomatis(seri, S_MODEL, pakai_uji=not TANPA_UJI)
awal_latih = _periode(AWAL_LATIH, _awal)
akhir_uji = None if TANPA_UJI else _periode(AKHIR_UJI, _akhir_uji)
akhir_latih = _periode(AKHIR_LATIH, _akhir_latih)

for _g in dp.cek_transformasi(seri.loc[awal_latih:], TRANSFORMASI):
    raise SystemExit("❌ " + _g)
if (_h := dp.petunjuk_log_ganda(seri, TRANSFORMASI)):
    print("⚠️", _h)

spek = am.SpesifikasiModel(order=(p, d, q), seasonal_order=(P, D, Q, S_MODEL), konstanta=KONSTANTA,
                           paksa_stasioner=PAKSA_STASIONER, paksa_invertible=PAKSA_INVERTIBLE)
pengaturan = pl.Pengaturan(spek=spek, transformasi=TRANSFORMASI, awal_latih=awal_latih, akhir_latih=akhir_latih,
                           akhir_uji=akhir_uji, horizon=HORIZON_MODEL, tingkat_kepercayaan=int(TINGKAT_KEPERCAYAAN),
                           koreksi_bias=KOREKSI_BIAS, kolom_x=tuple(kolom_x), transformasi_x=TRANSFORMASI_X,
                           lag_x=LAG_MODEL, proyeksi_x_sarima=PROYEKSI_X_SARIMA)

if PILIH_ORDE_OTOMATIS:
    _, _Xd, _gx = pl.siapkan_desain_x(X_dasar, pengaturan)
    if _gx:
        raise SystemExit("❌ " + "\n❌ ".join(_gx))
    _ym = dp.ke_skala_model(seri, TRANSFORMASI).loc[awal_latih:akhir_latih]
    print(f"Mencari orde terbaik ({KRITERIA}) pada data latih {dp.label_periode(awal_latih)} – "
          f"{dp.label_periode(akhir_latih)} ... (bisa memakan waktu beberapa menit)")
    tabel_orde, daftar_orde = am.pilih_orde(
        _ym, S_MODEL, X=_Xd, exog=tuple(_Xd.columns) if _Xd is not None else (), konstanta=KONSTANTA, d=d, D=D,
        maks_p=MAKS_p, maks_q=MAKS_q, maks_P=MAKS_P, maks_Q=MAKS_Q, kriteria=KRITERIA)
    _terbaik = daftar_orde[0]
    print(f"Jumlah model diestimasi: {len(tabel_orde)} | tidak konvergen: {int((~tabel_orde['Konvergen']).sum())} | "
          f"diestimasi dengan batasan stasioner: {int(tabel_orde['Batasan stasioner'].sum())}")
    print(f"Model terpilih: ARMA {tabel_orde.iloc[0]['ARMA (p,q)(P,Q)']} | {KRITERIA} = {tabel_orde.iloc[0][KRITERIA]:.6f}")
    display(tabel_orde.head(10))
    spek = _ganti(spek, order=_terbaik.order, seasonal_order=_terbaik.seasonal_order)
    pengaturan = _ganti(pengaturan, spek=spek)

_galat = pl.periksa(seri, pengaturan, X_dasar)
if _galat:
    raise SystemExit("❌ " + "\n❌ ".join(_galat))
print(f"✅ Model: {spek.label()}" + (f" + X ({', '.join(kolom_x)}; lag {LAG_MODEL}; transformasi X: {TRANSFORMASI_X})"
                                     if kolom_x else "") + f" | transformasi Y: {TRANSFORMASI}")
print(f"   Latih: {dp.label_periode(awal_latih)} – {dp.label_periode(akhir_latih)}"
      + (f" | Uji: {dp.label_periode(akhir_latih + 1)} – {dp.label_periode(akhir_uji)}" if akhir_uji else " | tanpa uji")
      + f" | Horizon: {HORIZON_MODEL}")
if KONSTANTA and (d or D):
    print("ℹ️ Dengan differencing, konstanta C berarti drift (tren rata-rata per periode); X juga ikut di-differencing.")


In [ ]:
#@title Langkah 5 — Jalankan model
try:
    hasil = pl.jalankan(seri, pengaturan, X_dasar)
except pl.GalatPengaturan as e:
    raise SystemExit("❌ " + str(e))

for _p in hasil.diagnostik_akhir["peringatan"]:
    print("⚠️", _p)
for _p in hasil.peringatan:
    print("ℹ️", _p)
if hasil.info_x:
    display(pd.Series(hasil.info_x, name="Keterangan").to_frame())

def _tampil_model(judul, m):
    display(Markdown(f"#### {judul}\n`{m.spek.label()}` — sampel {dp.label_periode(m.y_model.index[0])} s.d. "
                     f"{dp.label_periode(m.y_model.index[-1])}"))
    display(am.tabel_koefisien(m))
    display(am.statistik_ringkas(m).to_frame("Nilai"))

if hasil.backtest is not None:
    display(Markdown("### Model backtest — hanya data latih\nDipakai untuk meramal periode uji."))
    _tampil_model("Dengan X" if hasil.pakai_x else "Model", hasil.backtest["hasil"])
    if hasil.pakai_x:
        _tampil_model("Tanpa X (sampel sama)", hasil.backtest_tanpa_x["hasil"])
        display(Markdown("#### Perbandingan model backtest: dengan vs tanpa X"))
        display(pl.tabel_perbandingan_model(hasil, pakai_backtest=True))

display(Markdown(f"### Model untuk proyeksi ke depan — data s.d. {dp.label_periode(hasil.akhir_data_proyeksi)}"))
_tampil_model("Dengan X" if hasil.pakai_x else "Model", hasil.hasil_akhir)
if hasil.pakai_x:
    _tampil_model("Tanpa X (sampel sama)", hasil.hasil_tanpa_x)
    display(Markdown("#### Perbandingan model proyeksi: dengan vs tanpa X"))
    display(pl.tabel_perbandingan_model(hasil, pakai_backtest=False))


In [ ]:
#@title Langkah 6 — Diagnostik residual (model proyeksi)
_dg = hasil.diagnostik_akhir
print("Ljung-Box (p > 0,05 = tidak ada autokorelasi residual tersisa):")
display(_dg["ljung_box"].rename(columns={"lb_stat": "Statistik Q", "lb_pvalue": "p-value"}))
_jb = _dg["jarque_bera"]
print(f"Jarque-Bera = {_jb['statistik']:.3f}, p = {_jb['p_value']:.4f} → residual "
      f"{'normal' if _jb['p_value'] > 0.05 else 'TIDAK normal'}")
if _dg["akar_ar_invers_maks"] is not None:
    print(f"Akar AR invers maksimum = {_dg['akar_ar_invers_maks']:.4f} (harus < 1)")
if _dg["akar_ma_invers_maks"] is not None:
    print(f"Akar MA invers maksimum = {_dg['akar_ma_invers_maks']:.4f} (harus < 1)")
vz.grafik_acf(_dg).show()

In [ ]:
#@title Langkah 7 — Grafik & tabel hasil
vz.grafik_hasil(hasil, KOLOM_TARGET).show()
tabel = pl.tabel_hasil(hasil)
display(tabel.style.format({c: "{:,.2f}" for c in tabel.columns if c != "Jenis"}, na_rep="—"))
if hasil.pakai_x:
    display(Markdown("### Perbandingan ramalan: ex-post, ex-ante, dan tanpa X"))
    _tp = pl.tabel_perbandingan_ramalan(hasil)
    display(_tp.style.format({c: "{:,.2f}" for c in _tp.columns if c != "Jenis"}, na_rep="—"))


In [ ]:
#@title Langkah 8 — Evaluasi periode uji
if hasil.akurasi is None:
    print("Tidak ada periode uji (TANPA_UJI dicentang), jadi akurasi tidak dapat diukur.")
else:
    _teks = ("Akurasi **di luar sampel**: model hanya dilatih dengan data latih. "
             "Pembanding: *Naif musiman* (nilai musim lalu), *Rata-rata musiman*, dan "
             "*Naif (nilai terakhir)*. Pembanding musiman hanya muncul bila s ≥ 2.")
    if hasil.pakai_x:
        _teks += ("\n\n**Dengan X:** *ex-post* memakai X aktual periode uji (seolah X sudah diketahui), "
                  "*ex-ante* memakai X hasil proyeksi SARIMA dari data latih (kondisi nyata). "
                  "Baris *tanpa X* adalah model yang sama tanpa variabel X, pada sampel yang sama.")
    display(Markdown(_teks))
    display(hasil.akurasi)
    _nol = int(hasil.akurasi.iloc[0].get("Aktual = 0 (dikecualikan dari MAPE)", 0))
    if _nol:
        print(f"⚠️ {_nol} periode uji bernilai aktual 0 dan dikecualikan dari MAPE.")
    display(Markdown("\n".join(f"- **{k}**: {v}" for k, v in ev.PENJELASAN.items())))


In [ ]:
#@title Langkah 9 — Unduh hasil
#@markdown Pilih file yang ingin diunduh.
LAPORAN_EXCEL = False #@param {type:"boolean"}
HASIL_CSV = False #@param {type:"boolean"}
GRAFIK_PNG = False #@param {type:"boolean"}
RINGKASAN_TXT = False #@param {type:"boolean"}

import re
from google.colab import files
_dasar = re.sub(r"[^A-Za-z0-9_-]+", "_", f"proyeksi_{KOLOM_TARGET}").strip("_")
_keluaran = {}
if LAPORAN_EXCEL: _keluaran[f"{_dasar}_laporan.xlsx"] = er.excel_laporan(hasil)
if HASIL_CSV: _keluaran[f"{_dasar}.csv"] = er.csv_hasil(hasil)
if GRAFIK_PNG: _keluaran[f"{_dasar}.png"] = vz.png_hasil(hasil, KOLOM_TARGET)
if RINGKASAN_TXT: _keluaran[f"{_dasar}_ringkasan.txt"] = er.teks_ringkasan(hasil)
(FOLDER / "hasil").mkdir(exist_ok=True)
for _nama, _isi in _keluaran.items():
    _path = FOLDER / "hasil" / _nama
    _path.write_bytes(_isi)
    files.download(str(_path))
    print("⬇️", _nama)

---
## Opsional

**Uji otomatis** memeriksa bahwa mesin bekerja benar di Colab: pembacaan data dan frekuensi, likelihood eksak
(dengan dan tanpa X), differencing/drift, SARIMAX ex-post/ex-ante, pemilihan orde otomatis, model tanpa musim,
backtest tanpa kebocoran data uji, dan ekspor file.

**Tampilan web** menjalankan *ARIMA Forecasting Platform* dari Colab dengan tautan sementara
(saat ini untuk SARIMA tanpa X; dukungan X di tampilan web menyusul).
⚠️ Tautan `...trycloudflare.com` bersifat **publik** selama berjalan: jangan dibagikan bila datanya rahasia.
Untuk menutupnya: **Runtime → Disconnect and delete runtime**.


In [ ]:
#@title Uji otomatis (opsional)
TES = {
    'tests/test_modules.py': '"""\nUji modul. Jalankan dari folder proyek:  python -m pytest -q\nTes yang membutuhkan statsmodels otomatis dilewati bila statsmodels belum terpasang.\n"""\nimport io\nimport sys\nimport types\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pytest\n\nAKAR = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(AKAR))\n\nfrom modules import data_loader as dl  # noqa: E402\nfrom modules import data_preprocessing as dp  # noqa: E402\nfrom modules import evaluation as ev  # noqa: E402\nfrom modules import exploration as ex  # noqa: E402\nfrom modules import forecasting as fc  # noqa: E402\nfrom modules.arima_model import SpesifikasiModel, nama_parameter, peringatan_model  # noqa: E402\n\nCONTOH_XLSX = AKAR / "sample_data" / "sample_dataset.xlsx"\nCONTOH_CSV = AKAR / "sample_data" / "sample_dataset.csv"\n\n\n# ------------------------------------------------------------------ 1-2. file valid\n@pytest.mark.parametrize("path", [CONTOH_XLSX, CONTOH_CSV])\ndef test_file_contoh_terbaca(path):\n    df = dl.baca_file(str(path))\n    assert list(df.columns) == ["tanggal", "nilai", "dummy_libur", "indeks_harga"]\n    tgl, target = dl.tebak_kolom(df)\n    assert (tgl, target) == ("tanggal", "nilai")\n    frek = dp.tebak_frekuensi(df[tgl])\n    assert frek == "Bulanan"\n    v = dp.siapkan_seri(df, tgl, target, frek)\n    assert v.ok and len(v.peringatan) == 1 and "masa depan" in v.peringatan[0]\n    assert len(v.seri) == len(df) and str(v.seri.index.freq) in ("<MonthEnd>", "M")\n    assert dp.kolom_x_otomatis(df, tgl, target) == ["dummy_libur", "indeks_harga"]\n\n\ndef test_csv_titik_koma_dan_format_indonesia():\n    teks = "periode;nilai\\n2021M01;1.234.567,5\\n2021M02;200.000\\n2021M03;300000\\n"\n    f = io.StringIO(teks)\n    f.name = "x.csv"\n    df = dl.baca_file(f)\n    v = dp.siapkan_seri(df, "periode", "nilai", "Bulanan")\n    assert v.seri.tolist() == [1234567.5, 200000.0, 300000.0]\n\n\ndef test_format_file_salah():\n    with pytest.raises(dl.GalatData):\n        dl.baca_file("data.txt")\n\n\n# ------------------------------------------------------------------ 3. nilai kosong\ndef _df(tanggal, nilai):\n    return pd.DataFrame({"t": tanggal, "v": nilai})\n\n\ndef test_periode_hilang_dilaporkan_tidak_dihapus():\n    per = [f"2021M{m:02d}" for m in range(1, 13) if m != 5]\n    v = dp.siapkan_seri(_df(per, np.arange(1.0, 12.0)), "t", "v", "Bulanan")\n    assert v.ok\n    assert len(v.seri) == 12 and np.isnan(v.seri.loc["2021-05"])\n    assert any("tidak ada di data" in p for p in v.peringatan)\n\n\ndef test_nilai_kosong_dilaporkan():\n    per = [f"2021M{m:02d}" for m in range(1, 13)]\n    nilai = [1.0] * 12\n    nilai[3] = None\n    v = dp.siapkan_seri(_df(per, nilai), "t", "v", "Bulanan")\n    assert v.ok and v.seri.isna().sum() == 1\n    assert any("kosong" in p for p in v.peringatan)\n\n\n# ------------------------------------------------------------------ 4. tanggal tidak valid\ndef test_tanggal_tidak_valid():\n    v = dp.siapkan_seri(_df(["abc", "def"] * 6, range(12)), "t", "v", "Bulanan")\n    assert not v.ok and "tidak bisa dibaca" in v.galat[0]\n\n\ndef test_periode_ganda_salah_frekuensi():\n    harian = pd.date_range("2021-01-01", periods=40, freq="D")\n    v = dp.siapkan_seri(_df(harian, range(40)), "t", "v", "Bulanan")\n    assert not v.ok and "periode ganda" in v.galat[0]\n\n\ndef test_tanggal_format_indonesia_hari_dulu():\n    v = dp.siapkan_seri(_df([f"01/{m:02d}/2021" for m in range(2, 13)], range(1, 12)), "t", "v", "Bulanan")\n    assert v.ok, v.galat\n    assert dp.label_periode(v.seri.index[0]) == "2021-02" and len(v.seri) == 11\n\n\n# ------------------------------------------------------------------ 5. target bukan angka\ndef test_target_bukan_angka():\n    per = [f"2021M{m:02d}" for m in range(1, 13)]\n    v = dp.siapkan_seri(_df(per, ["tinggi", "rendah"] * 6), "t", "v", "Bulanan")\n    assert not v.ok and "bukan angka" in v.galat[0]\n\n\ndef test_kolom_sama():\n    v = dp.siapkan_seri(_df(["2021M01"], [1]), "t", "t", "Bulanan")\n    assert not v.ok\n\n\n# ------------------------------------------------------------------ 6. observasi kurang\ndef test_observasi_terlalu_sedikit():\n    per = [f"2021M{m:02d}" for m in range(1, 6)]\n    v = dp.siapkan_seri(_df(per, range(1, 6)), "t", "v", "Bulanan")\n    assert not v.ok and "terlalu sedikit" in v.galat[0]\n\n\ndef test_minimal_observasi_model_musiman():\n    assert SpesifikasiModel(seasonal_order=(1, 0, 0, 12)).minimal_observasi() >= 18\n    assert SpesifikasiModel(order=(1, 0, 0), seasonal_order=(0, 0, 0, 12)).minimal_observasi() == 10\n\n\n# ------------------------------------------------------------------ 7. nol/negatif dengan log\ndef test_log_menolak_nol_negatif():\n    s = pd.Series([1.0, 0.0, -2.0, 4.0], index=pd.period_range("2021-01", periods=4, freq="M"))\n    assert dp.cek_transformasi(s, "log")\n    assert not dp.cek_transformasi(s, "tanpa")\n\n\ndef test_log_dan_kebalikannya():\n    s = pd.Series([100.0, 200.0, 50.0], index=pd.period_range("2021-01", periods=3, freq="M"))\n    for mode in dp.TRANSFORMASI:\n        balik = dp.ke_skala_asli(dp.ke_skala_model(s, mode), mode)\n        harapan = np.exp(s) if mode == "sudah_log" else s\n        np.testing.assert_allclose(balik, harapan)\n\n\ndef test_petunjuk_log_ganda():\n    s = pd.Series(np.linspace(11.5, 13.5, 24), index=pd.period_range("2021-01", periods=24, freq="M"))\n    assert dp.petunjuk_log_ganda(s, "log")\n    assert dp.petunjuk_log_ganda(s * 1e5, "log") is None\n\n\n# ------------------------------------------------------------------ frekuensi & periode otomatis\n@pytest.mark.parametrize("tanggal,harapan", [\n    (pd.date_range("2020-01-31", periods=24, freq="ME"), "Bulanan"),\n    (pd.date_range("2020-03-31", periods=12, freq="QE"), "Triwulanan"),\n    (pd.date_range("2024-01-01", periods=30, freq="W-MON"), "Mingguan"),\n    (pd.date_range("2024-01-01", periods=30, freq="D"), "Harian"),\n    ([f"2021M{m:02d}" for m in range(1, 13)], "Bulanan"),\n    (["2020Q1", "2020Q2", "2020Q3"], "Triwulanan"),\n    ([2001, 2002, 2003], "Tahunan"),\n])\ndef test_tebak_frekuensi(tanggal, harapan):\n    assert dp.tebak_frekuensi(pd.Series(tanggal)) == harapan\n\n\ndef test_periode_otomatis():\n    from modules.pipeline import periode_otomatis\n    seri = pd.Series(np.arange(60.0), index=pd.period_range("2015-01", periods=60, freq="M"))\n    awal, akhir_latih, akhir_uji = periode_otomatis(seri, 12)\n    assert (awal, akhir_latih, akhir_uji) == (seri.index[0], seri.index[-13], seri.index[-1])\n    assert periode_otomatis(seri, 12, pakai_uji=False)[2] is None\n    pendek = seri.iloc[:30]\n    assert periode_otomatis(pendek, 12)[1] == pendek.index[-7]  # 20% data\n\n\n# ------------------------------------------------------------------ frekuensi lain\n@pytest.mark.parametrize("frek,tanggal,n", [\n    ("Triwulanan", ["2020Q1", "2020Q2", "2020Q3", "2020Q4"] * 1 + ["2021Q1", "2021Q2", "2021Q3", "2021Q4",\n                                                                     "2022Q1", "2022Q2", "2022Q3"], 11),\n    ("Mingguan", pd.date_range("2024-01-01", periods=30, freq="W-MON"), 30),\n    ("Harian", pd.date_range("2024-01-01", periods=30, freq="D"), 30),\n])\ndef test_frekuensi_lain(frek, tanggal, n):\n    v = dp.siapkan_seri(_df(tanggal, np.arange(1.0, n + 1)), "t", "v", frek)\n    assert v.ok, v.galat\n    assert len(v.seri) == n\n    assert dp.label_periode(v.seri.index[0])\n\n\n# ------------------------------------------------------------------ evaluasi\ndef test_metrik_definisi():\n    a, f = [100, 200, 150], [110, 180, 150]\n    m = ev.ukuran_akurasi(a, f)\n    assert m["MAE"] == pytest.approx(10.0)\n    assert m["RMSE"] == pytest.approx(np.sqrt((100 + 400 + 0) / 3))\n    assert m["MAPE (%)"] == pytest.approx((10 / 100 + 20 / 200 + 0) / 3 * 100)\n    assert m["Theil U2"] == pytest.approx(0.1940285, rel=1e-6)\n\n\ndef test_mape_mengecualikan_nol():\n    m = ev.ukuran_akurasi([0, 100], [10, 110])\n    assert m["MAPE (%)"] == pytest.approx(10.0)\n    assert m["Aktual = 0 (dikecualikan dari MAPE)"] == 1\n    assert m["MAE"] == pytest.approx(10.0)\n\n\n# ------------------------------------------------------------------ baseline\ndef test_baseline_musiman():\n    idx = pd.period_range("2010-01", periods=36, freq="M")\n    y = pd.Series(np.arange(36.0), index=idx)\n    latih, uji = y.iloc[:24], idx[24:]\n    np.testing.assert_allclose(fc.baseline_naif_musiman(latih, uji, 12), y.iloc[12:24].to_numpy())\n    np.testing.assert_allclose(fc.baseline_rata_musiman(latih, uji, 12), (y.iloc[:12].to_numpy() + y.iloc[12:24].to_numpy()) / 2)\n\n\n# ------------------------------------------------------------------ peringatan model\ndef test_peringatan_model_degeneratif():\n    params = pd.Series({"C": 10.0, "ar.S.L12": 1.0, "ma.S.L12": -0.999895, "sigma2": 0.4})\n    bse = pd.Series({"C": 0.22, "ar.S.L12": 7.39e-5, "ma.S.L12": 9.3e-5, "sigma2": 0.09})\n    pesan = peringatan_model(params, bse, np.ones(12), np.ones(12) * 0.99999, 12)\n    assert len(pesan) == 4\n    params2 = pd.Series({"C": 10.0, "ar.S.L12": 0.45, "sigma2": 0.4})\n    assert peringatan_model(params2, params2 * 0 + 0.1, np.full(12, 0.45 ** (1 / 12)), np.array([]), 12) == []\n\n\ndef test_nama_parameter():\n    assert [nama_parameter(n) for n in ["ar.S.L12", "ma.L1", "sigma2", "C"]] == ["SAR(12)", "MA(1)", "SIGMASQ", "C"]\n\n\n# ------------------------------------------------------------------ eksplorasi & PNG\ndef test_statistik_deskriptif():\n    s = pd.Series([1.0, 2.0, None, 4.0], index=pd.period_range("2021-01", periods=4, freq="M"))\n    st = ex.statistik_deskriptif(s)\n    assert st["Observasi terisi"] == 3 and st["Median"] == 2.0\n    assert ex.daftar_kosong(s) == ["2021-03"]\n\n\ndef test_png_dari_hasil_tiruan():\n    from modules.visualization import png_hasil\n    idx = pd.period_range("2021-01", periods=24, freq="M")\n    y = pd.Series(np.arange(24.0) + 10, index=idx)\n    uji = idx[18:]\n    depan = pd.period_range("2023-01", periods=6, freq="M")\n    h = types.SimpleNamespace(\n        pengaturan=types.SimpleNamespace(awal_latih=idx[0], akhir_latih=idx[17], akhir_uji=idx[-1],\n                                         tingkat_kepercayaan=95, spek=SpesifikasiModel()),\n        y_asli=y, akhir_data_proyeksi=idx[-1],\n        backtest={"tabel": pd.DataFrame({"aktual": y[uji], "ramalan": y[uji] + 1, "batas_bawah": y[uji] - 2,\n                                         "batas_atas": y[uji] + 3}, index=uji)},\n        ramalan=pd.DataFrame({"ramalan": np.ones(6) * 30, "batas_bawah": np.ones(6) * 25,\n                              "batas_atas": np.ones(6) * 35}, index=depan),\n    )\n    png = png_hasil(h, "nilai")\n    assert png[:8] == b"\\x89PNG\\r\\n\\x1a\\n" and len(png) > 10_000\n\n\n# ------------------------------------------------------------------ 8-10. butuh statsmodels\n@pytest.fixture\ndef seri_sar12():\n    """Proses SAR(12) sintetis: log y = 12,6 + u, u_t = 0,5 u_{t-12} + e."""\n    rng = np.random.default_rng(42)\n    n = 240\n    u = np.zeros(n + 120)\n    e = rng.normal(0, 0.3, n + 120)\n    for t in range(12, n + 120):\n        u[t] = 0.5 * u[t - 12] + e[t]\n    idx = pd.period_range("2000-01", periods=n, freq="M")\n    return pd.Series(np.exp(12.6 + u[120:]), index=idx, name="nilai")\n\n\ndef _pengaturan(seri, horizon=12, spek=None, uji=True):\n    from modules.pipeline import Pengaturan\n    return Pengaturan(spek=spek or SpesifikasiModel(), transformasi="log", awal_latih=seri.index[0],\n                      akhir_latih=seri.index[-25], akhir_uji=seri.index[-1] if uji else None, horizon=horizon)\n\n\ndef test_estimasi_memulihkan_parameter(seri_sar12):\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import jalankan\n    h = jalankan(seri_sar12, _pengaturan(seri_sar12))\n    k = h.koefisien_akhir["Koefisien"]\n    assert k["C"] == pytest.approx(12.6, abs=0.1)\n    assert k["SAR(12)"] == pytest.approx(0.5, abs=0.15)\n    assert h.akurasi is not None and len(h.backtest["tabel"]) == 24\n\n\n@pytest.mark.parametrize("horizon", [1, 12, 36])\ndef test_horizon(seri_sar12, horizon):\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import jalankan\n    h = jalankan(seri_sar12, _pengaturan(seri_sar12, horizon, uji=False))\n    assert len(h.ramalan) == horizon and h.backtest is None\n    assert h.ramalan.index[0] == seri_sar12.index[-1] + 1\n    assert (h.ramalan["batas_bawah"] < h.ramalan["ramalan"]).all()\n    assert (h.ramalan["ramalan"] < h.ramalan["batas_atas"]).all()\n\n\ndef test_backtest_tanpa_kebocoran(seri_sar12):\n    """Mengubah nilai periode uji tidak boleh mengubah ramalan backtest."""\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import jalankan\n    peng = _pengaturan(seri_sar12)\n    a = jalankan(seri_sar12, peng).backtest["tabel"]["ramalan"]\n    rusak = seri_sar12.copy()\n    rusak.iloc[-24:] *= 5\n    b = jalankan(rusak, peng).backtest["tabel"]["ramalan"]\n    np.testing.assert_allclose(a, b)\n\n\ndef test_likelihood_eksak(seri_sar12):\n    """LL = log-likelihood Gaussian eksak (matriks kovarians penuh), tanpa burn-in."""\n    pytest.importorskip("statsmodels")\n    from scipy.linalg import toeplitz\n    from scipy.stats import multivariate_normal\n    from statsmodels.tsa.arima_process import arma_acovf\n    from modules import arima_model as am\n    y = np.log(seri_sar12.iloc[:120])\n    m = am.estimasi(y, SpesifikasiModel())\n    phi, c, s2 = m.params["ar.S.L12"], m.params["C"], m.params["sigma2"]\n    gamma = arma_acovf(np.r_[1, np.zeros(11), -phi], [1], nobs=len(y), sigma2=s2)\n    ll = multivariate_normal(mean=np.full(len(y), c), cov=toeplitz(gamma)).logpdf(y.to_numpy())\n    assert m.llf == pytest.approx(ll, rel=1e-7)\n    assert m.nobs == len(y)\n    st = am.statistik_ringkas(m)\n    assert st["Akaike info criterion (AIC)"] == pytest.approx((-2 * ll + 2 * 3) / len(y), rel=1e-6)\n\n\ndef _integrasi_manual(m, langkah):\n    """Ramalan level dari ramalan model ARMA pada data hasil differencing, diintegrasikan manual."""\n    spek = m.spek\n    d = spek.order[1]\n    _, D, _, s = spek.seasonal_order\n    exog = np.ones((langkah, 1)) if spek.konstanta else None\n    f = np.asarray(m.res.get_forecast(langkah, exog=exog).predicted_mean)\n    y = list(m.y_model.to_numpy())\n    # hanya untuk satu jenis differencing per tes\n    for h in range(langkah):\n        if d == 1:\n            y.append(y[-1] + f[h])\n        else:\n            y.append(y[-s] + f[h])\n    return np.array(y[-langkah:])\n\n\n@pytest.mark.parametrize("order,seasonal", [((1, 1, 0), (0, 0, 0, 12)), ((0, 0, 1), (0, 1, 0, 12))])\ndef test_differencing_dan_drift(order, seasonal):\n    """Dengan differencing, C = drift dan ramalan level = integrasi ramalan data hasil differencing."""\n    pytest.importorskip("statsmodels")\n    from modules import arima_model as am\n    rng = np.random.default_rng(3)\n    idx = pd.period_range("2005-01", periods=150, freq="M")\n    musim = np.tile(np.sin(np.arange(12) / 12 * 2 * np.pi), 13)[:150]\n    y = pd.Series(10 + np.cumsum(0.02 + rng.normal(0, 0.05, 150)) + musim, index=idx)\n    spek = SpesifikasiModel(order=order, seasonal_order=seasonal)\n    m = am.estimasi(y, spek)\n    assert m.nobs == len(y) - spek.observasi_hilang()\n    ramal = fc.proyeksi(m, 24, "tanpa")\n    np.testing.assert_allclose(ramal["ramalan"].to_numpy(), _integrasi_manual(m, 24), rtol=1e-6)\n    assert (ramal["batas_atas"] > ramal["batas_bawah"]).all()\n    lebar = (ramal["batas_atas"] - ramal["batas_bawah"]).to_numpy()\n    assert lebar[-1] > lebar[0]  # ketidakpastian bertambah dengan horizon\n\n\ndef test_tanpa_musim_tahunan():\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import Pengaturan, jalankan\n    rng = np.random.default_rng(5)\n    e = rng.normal(0, 1, 60)\n    u = np.zeros(60)\n    for t in range(1, 60):\n        u[t] = 0.6 * u[t - 1] + e[t]\n    df = _df([str(t) for t in range(1965, 2025)], 50 + u)\n    assert dp.tebak_frekuensi(df["t"]) == "Tahunan"\n    seri = dp.siapkan_seri(df, "t", "v", "Tahunan").seri\n    spek = SpesifikasiModel(order=(1, 0, 0), seasonal_order=(0, 0, 0, 1))\n    peng = Pengaturan(spek=spek, transformasi="tanpa", awal_latih=seri.index[0], akhir_latih=seri.index[-6],\n                      akhir_uji=seri.index[-1], horizon=5)\n    h = jalankan(seri, peng)\n    assert spek.label() == "ARIMA(1,0,0) + C"\n    assert h.koefisien_akhir.loc["AR(1)", "Koefisien"] == pytest.approx(0.6, abs=0.25)\n    assert "Naif (nilai terakhir)" in h.akurasi.index and "Naif musiman" not in h.akurasi.index\n    assert dp.label_periode(h.ramalan.index[0]) == "2025"\n\n\ndef test_musiman_butuh_s_lebih_dari_1():\n    from modules.pipeline import Pengaturan, periksa\n    seri = pd.Series(np.arange(1.0, 41.0), index=pd.period_range("1980", periods=40, freq="Y"))\n    peng = Pengaturan(spek=SpesifikasiModel(seasonal_order=(1, 0, 0, 1)), transformasi="tanpa",\n                      awal_latih=seri.index[0], akhir_latih=seri.index[-6], akhir_uji=seri.index[-1])\n    assert any("s ≥ 2" in g for g in periksa(seri, peng))\n\n\ndef test_model_manual_lain_dan_ekspor(seri_sar12):\n    pytest.importorskip("statsmodels")\n    from modules import export_results as exr\n    from modules.pipeline import jalankan, tabel_hasil\n    h = jalankan(seri_sar12, _pengaturan(seri_sar12, spek=SpesifikasiModel(order=(1, 0, 1),\n                                                                         seasonal_order=(1, 0, 0, 12))))\n    assert {"AR(1)", "MA(1)", "SAR(12)", "C", "SIGMASQ"} <= set(h.koefisien_akhir.index)\n    assert len(tabel_hasil(h)) == 24 + 12\n    xls = pd.read_excel(io.BytesIO(exr.excel_laporan(h)), sheet_name=None)\n    assert {"Pengaturan", "Hasil", "Evaluasi", "Koefisien", "Statistik model"} <= set(xls)\n    assert exr.csv_hasil(h).startswith("\\ufeffPeriode".encode("utf-8"))\n    assert b"MODEL UNTUK PROYEKSI" in exr.teks_ringkasan(h)\n\n\n# ------------------------------------------------------------------ SARIMAX (variabel independen)\ndef _data_sarimax(n=120, beta=(1.3, 0.8), phi=0.5, c=10.0, seed=11, horizon_x=0):\n    """y = C + b1·D1 + b2·X2 + u, u_t = phi·u_{t-12} + e. D1 dummy, X2 kontinu. Bisa diberi baris X masa depan."""\n    rng = np.random.default_rng(seed)\n    total = n + horizon_x\n    idx = pd.period_range("2010-01", periods=total, freq="M")\n    d1 = (rng.random(total) < 0.15).astype(float)\n    x2 = np.cumsum(rng.normal(0, 0.3, total)) * 0.2 + rng.normal(0, 1, total)\n    u = np.zeros(total + 120)\n    e = rng.normal(0, 0.4, total + 120)\n    for t in range(12, total + 120):\n        u[t] = phi * u[t - 12] + e[t]\n    y = c + beta[0] * d1 + beta[1] * x2 + u[120:]\n    y[n:] = np.nan\n    X = pd.DataFrame({"D1": d1, "X2": x2}, index=idx)\n    return pd.Series(y, index=idx, name="y"), X\n\n\ndef test_siapkan_x_dan_transformasi():\n    df = pd.DataFrame({"tgl": [f"2021M{m:02d}" for m in range(1, 13)],\n                       "y": [5.0] * 10 + [None, None],\n                       "dummy": [0, 1] * 6, "harga": np.arange(1.0, 13.0), "teks": ["a"] * 12})\n    assert dp.kolom_x_otomatis(df, "tgl", "y") == ["dummy", "harga"]\n    v = dp.siapkan_seri(df, "tgl", "y", "Bulanan")\n    assert v.ok and any("masa depan" in p for p in v.peringatan) and not any("nilai target kosong" in p for p in v.peringatan)\n    vx = dp.siapkan_x(df, "tgl", ["dummy", "harga"], "Bulanan", v.seri.index)\n    assert vx.ok and len(vx.X) == 12 and vx.X["harga"].iloc[-1] == 12.0\n    Xm, g = dp.transformasi_x(vx.X, "log")\n    assert list(Xm.columns) == ["dummy", "log(harga)"] and not g  # dummy tidak di-log\n    assert list(dp.bentuk_lag(Xm, (0, 1)).columns) == ["dummy", "dummy(-1)", "log(harga)", "log(harga)(-1)"]\n    _, g = dp.transformasi_x(pd.DataFrame({"z": [1.0, -1.0, 2.0]}), "log")\n    assert g\n\n\ndef test_likelihood_eksak_dengan_x():\n    pytest.importorskip("statsmodels")\n    from scipy.linalg import toeplitz\n    from scipy.stats import multivariate_normal\n    from statsmodels.tsa.arima_process import arma_acovf\n    from modules import arima_model as am\n    y, X = _data_sarimax()\n    spek = SpesifikasiModel(exog=("D1", "X2"))\n    m = am.estimasi(y, spek, X)\n    p = m.params\n    assert p["D1"] == pytest.approx(1.3, abs=0.3) and p["X2"] == pytest.approx(0.8, abs=0.15)\n    rata = p["C"] + p["D1"] * X["D1"].iloc[:120] + p["X2"] * X["X2"].iloc[:120]\n    gamma = arma_acovf(np.r_[1, np.zeros(11), -p["ar.S.L12"]], [1], nobs=120, sigma2=p["sigma2"])\n    ll = multivariate_normal(mean=rata.to_numpy(), cov=toeplitz(gamma)).logpdf(y.iloc[:120].to_numpy())\n    assert m.llf == pytest.approx(ll, rel=1e-7)\n    st = am.statistik_ringkas(m)\n    assert st["Sum squared resid"] == pytest.approx(m.nobs * p["sigma2"], rel=1e-4)\n    n, k = m.nobs, len(p)\n    assert st["Adjusted R-squared"] == pytest.approx(1 - (1 - st["R-squared"]) * (n - 1) / (n - k))\n    assert st["S.E. of regression"] == pytest.approx(np.sqrt(st["Sum squared resid"] / (n - k)))\n    tk = am.tabel_koefisien(m)\n    assert list(tk.index) == ["C", "D1", "X2", "SAR(12)", "SIGMASQ"]\n    from scipy import stats as sst\n    assert tk.loc["X2", "Prob."] == pytest.approx(2 * sst.t.sf(abs(tk.loc["X2", "t-Statistic"]), n - k))\n\n\ndef test_sarimax_differencing_ramalan_level():\n    """d = 1 dengan X: Δy = C + βΔx + u; ramalan level = integrasi manual."""\n    pytest.importorskip("statsmodels")\n    from modules import arima_model as am\n    y, X = _data_sarimax(horizon_x=12)\n    y = y.cumsum() / 10\n    spek = SpesifikasiModel(order=(1, 1, 0), seasonal_order=(0, 0, 0, 12), exog=("X2",))\n    m = am.estimasi(y.iloc[:120], spek, X)\n    assert m.nobs == 119\n    ramal = fc.proyeksi(m, 12, "tanpa", X_depan=X.iloc[120:])\n    dx = X["X2"].diff().iloc[120:].to_numpy()\n    f = np.asarray(m.res.get_forecast(12, exog=np.column_stack([np.ones(12), dx])).predicted_mean)\n    np.testing.assert_allclose(ramal["ramalan"].to_numpy(), y.iloc[119] + np.cumsum(f), rtol=1e-6)\n    with pytest.raises(ValueError):\n        fc.proyeksi(m, 12, "tanpa")  # X masa depan wajib\n\n\ndef test_pipeline_sarimax_bandingkan():\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import (NAMA_EX_ANTE, NAMA_EX_POST_DEPAN, NAMA_TANPA_X, Pengaturan, jalankan,\n                                  tabel_perbandingan_model, tabel_perbandingan_ramalan)\n    y, X = _data_sarimax(horizon_x=6)\n    peng = Pengaturan(spek=SpesifikasiModel(), transformasi="tanpa", awal_latih=y.index[0],\n                      akhir_latih=y.index[107], akhir_uji=y.index[119], horizon=6, kolom_x=("D1", "X2"),\n                      lag_x=(0,), maks_orde_x=(1, 0, 1, 0))\n    h = jalankan(y, peng, X)\n    assert h.pakai_x and h.spek.exog == ("D1", "X2")\n    baris = list(h.akurasi.index)\n    assert any("ex-post" in b for b in baris) and any("ex-ante" in b for b in baris) and any("tanpa X" in b for b in baris)\n    assert set(h.ramalan_varian) == {NAMA_EX_POST_DEPAN, NAMA_EX_ANTE, NAMA_TANPA_X}\n    assert h.nama_ramalan_utama == NAMA_EX_POST_DEPAN and len(h.ramalan) == 6\n    # sampel model tanpa X sama dengan model dengan X\n    assert h.hasil_tanpa_x.nobs == h.hasil_akhir.nobs\n    assert h.backtest_tanpa_x["hasil"].nobs == h.backtest["hasil"].nobs\n    assert len(tabel_perbandingan_ramalan(h)) == 12 + 6\n    assert tabel_perbandingan_model(h).shape[1] == 2\n    # model dengan X yang benar harus lebih akurat (ex-post) daripada tanpa X\n    mae = h.akurasi["MAE"]\n    assert mae[[b for b in baris if "ex-post" in b][0]] < mae[[b for b in baris if "tanpa X" in b][0]]\n\n\ndef test_pipeline_sarimax_lag_dan_tanpa_x_manual():\n    pytest.importorskip("statsmodels")\n    from modules.pipeline import NAMA_EX_ANTE, Pengaturan, jalankan\n    y, X = _data_sarimax()\n    peng = Pengaturan(spek=SpesifikasiModel(), transformasi="tanpa", awal_latih=y.index[0],\n                      akhir_latih=y.index[107], akhir_uji=y.index[119], horizon=4, kolom_x=("X2",),\n                      lag_x=(0, 1), maks_orde_x=(1, 0, 0, 0))\n    h = jalankan(y, peng, X)\n    assert h.spek.exog == ("X2", "X2(-1)")\n    assert h.hasil_akhir.nobs == 119  # satu observasi awal hilang karena lag\n    assert h.nama_ramalan_utama == NAMA_EX_ANTE  # tidak ada X manual ke depan\n    peng2 = Pengaturan(**{**peng.__dict__, "proyeksi_x_sarima": False})\n    from modules.pipeline import GalatPengaturan\n    with pytest.raises(GalatPengaturan):\n        jalankan(y, peng2, X)\n\n\ndef test_pilih_orde_otomatis():\n    pytest.importorskip("statsmodels")\n    from modules import arima_model as am\n    y, X = _data_sarimax(n=96)\n    tabel, daftar = am.pilih_orde(y.iloc[:96], 12, X=X, exog=("D1", "X2"), maks_p=2, maks_q=2, maks_P=1, maks_Q=1)\n    assert len(tabel) == 36 == len(daftar)\n    aic = tabel["AIC"].dropna().to_numpy()\n    assert (np.diff(aic) >= 0).all()\n    assert daftar[0].exog == ("D1", "X2")\n',
}
for _nama, _isi in TES.items():
    (FOLDER / _nama).parent.mkdir(parents=True, exist_ok=True)
    (FOLDER / _nama).write_text(_isi)
!cd /content/sarima_app && python -m pytest -q -p no:cacheprovider tests

In [ ]:
#@title Buka tampilan web (opsional)
BERKAS_WEB = {
    'app.py': '"""\nSARIMA / SARIMAX Forecasting Platform — titik masuk aplikasi.\nJalankan dari folder proyek:  python -m streamlit run app.py\n"""\nimport streamlit as st\n\nfrom halaman import komponen\n\nst.set_page_config(page_title="SARIMA / SARIMAX Forecasting Platform", page_icon="📈", layout="wide")\nkomponen.gaya()\n\nnavigasi = st.navigation({\n    "Mulai": [\n        st.Page("halaman/beranda.py", title="Beranda", icon=":material/home:", default=True),\n    ],\n    "Alur kerja": [\n        st.Page("halaman/unggah.py", title="1. Unggah Data", icon=":material/upload_file:"),\n        st.Page("halaman/eksplorasi.py", title="2. Eksplorasi Data", icon=":material/query_stats:"),\n        st.Page("halaman/konfigurasi.py", title="3. Konfigurasi & Jalankan", icon=":material/tune:"),\n        st.Page("halaman/hasil.py", title="4. Hasil Proyeksi", icon=":material/show_chart:"),\n        st.Page("halaman/evaluasi.py", title="5. Evaluasi Model", icon=":material/fact_check:"),\n        st.Page("halaman/unduh.py", title="6. Unduh Hasil", icon=":material/download:"),\n    ],\n})\nkomponen.status_sidebar()\nnavigasi.run()\n',
    'halaman/__init__.py': '',
    'halaman/beranda.py': 'import streamlit as st\n\nst.title("📈 SARIMA / SARIMAX Forecasting Platform")\nst.markdown("##### Platform interaktif untuk proyeksi data deret waktu")\n\nst.markdown(\n    """\nPlatform ini membantu Anda membuat **proyeksi (forecast)** dari data deret waktu, misalnya\narus kas, penjualan, produksi, atau harga, dengan model **ARIMA/SARIMA**, tanpa perlu menulis kode.\nBila ada variabel penjelas (mis. dummy hari raya, harga, indeks), gunakan **SARIMAX** dan bandingkan langsung\nhasilnya dengan model tanpa variabel tersebut.\n"""\n)\n\nwith st.expander("Apa itu ARIMA?", expanded=False):\n    st.markdown(\n        """\n**ARIMA** (*AutoRegressive Integrated Moving Average*) adalah model statistik yang meramal nilai\nke depan berdasarkan pola data itu sendiri:\n\n- **AR (p)**: nilai sekarang dipengaruhi nilai periode sebelumnya.\n- **I (d)**: berapa kali data dibedakan (*differencing*) agar stabil (stasioner).\n- **MA (q)**: nilai sekarang dipengaruhi kesalahan ramalan periode sebelumnya.\n\nBila ada pola yang berulang tiap tahun (musiman), dipakai **SARIMA** dengan komponen musiman\n(P, D, Q) dan panjang musim *s* (mis. 12 untuk data bulanan). Bila ditambah variabel independen (X),\nmodelnya disebut **SARIMAX**: X ikut menjelaskan naik-turunnya data, dan nilai X ke depan bisa diisi manual\n(*ex-post*) atau diproyeksikan dengan SARIMA (*ex-ante*).\n"""\n    )\n\nst.markdown("### Alur kerja")\nlangkah = [\n    ("1", "Unggah data", "File Excel/CSV: tanggal, nilai, dan (opsional) variabel X."),\n    ("2", "Eksplorasi", "Lihat grafik, statistik, dan nilai kosong."),\n    ("3", "Konfigurasi", "Transformasi, periode, X, orde (bisa otomatis)."),\n    ("4", "Hasil", "Koefisien model, grafik, dan tabel ramalan."),\n    ("5", "Evaluasi", "Akurasi periode uji; dengan vs tanpa X."),\n    ("6", "Unduh", "Excel, CSV, PNG, dan ringkasan teks."),\n]\nkolom = st.columns(len(langkah))\nfor k, (no, judul, isi) in zip(kolom, langkah):\n    k.markdown(f"<div class=\'langkah\'><b>{no}. {judul}</b><br><small>{isi}</small></div>", unsafe_allow_html=True)\n\nst.markdown("### Cara memakai")\nst.markdown(\n    """\n1. Buka **Unggah Data**, unggah file Anda (atau pakai data contoh), lalu pilih kolom tanggal, kolom yang ingin diramal, dan frekuensinya.\n2. Periksa data di **Eksplorasi Data**.\n3. Di **Konfigurasi & Jalankan**, atur model lalu klik **Jalankan Proyeksi**.\n4. Lihat hasilnya di **Hasil Proyeksi** dan **Evaluasi Model**, lalu unduh di **Unduh Hasil**.\n\nData yang Anda unggah hanya diproses selama sesi berjalan dan tidak disimpan oleh aplikasi.\n"""\n)\n\nif st.button("Mulai proyek baru", type="primary", icon=":material/arrow_forward:"):\n    st.switch_page("halaman/unggah.py")\n',
    'halaman/eksplorasi.py': 'import streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.data_preprocessing import FREKUENSI\nfrom modules.exploration import daftar_kosong, dekomposisi, statistik_deskriptif\nfrom modules.visualization import grafik_seri, grafik_x\n\nseri = kp.perlu_seri()\nseri = seri.loc[:seri.last_valid_index()]  # tanpa baris masa depan\ntarget = st.session_state[kp.TARGET]\n\nst.title("2. Eksplorasi Data")\nst.plotly_chart(grafik_seri(seri, f"{target} ({st.session_state[kp.FREKUENSI].lower()})", target))\n\nc1, c2 = st.columns([1, 1])\nwith c1:\n    st.subheader("Statistik deskriptif")\n    stat = statistik_deskriptif(seri)\n    tampil = {k: (kp.fmt_angka(v) if isinstance(v, float) else v) for k, v in stat.items()}\n    st.dataframe({"Ukuran": list(tampil), "Nilai": [str(v) for v in tampil.values()]}, hide_index=True)\nwith c2:\n    st.subheader("Nilai kosong")\n    kosong = daftar_kosong(seri)\n    if kosong:\n        st.warning(f"{len(kosong)} periode kosong: {\', \'.join(kosong)}. "\n                   "Periode ini tidak diisi; model memperlakukannya sebagai data hilang.")\n    else:\n        st.success("Tidak ada periode kosong.")\n\nX_dasar = st.session_state.get(kp.X_DASAR)\nif X_dasar is not None:\n    st.subheader("Variabel independen (X)")\n    st.caption("Termasuk nilai X periode ke depan bila diisi di file.")\n    st.plotly_chart(grafik_x(X_dasar, "Variabel X"))\n\nst.subheader("Dekomposisi (opsional)")\nst.caption("Memisahkan data menjadi tren, pola musiman, dan sisa. Hanya untuk informasi; tidak memengaruhi model.")\nwith st.expander("Tampilkan dekomposisi"):\n    s_bawaan = FREKUENSI[st.session_state[kp.FREKUENSI]][1]\n    c1, c2 = st.columns(2)\n    s = c1.number_input("Panjang musim", 2, 365, max(2, s_bawaan))\n    jenis = c2.radio("Jenis", ["additive", "multiplicative"], horizontal=True,\n                     format_func=lambda x: "Aditif" if x == "additive" else "Multiplikatif")\n    try:\n        d = dekomposisi(seri, int(s), jenis)\n        import plotly.graph_objects as go\n        from plotly.subplots import make_subplots\n        fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=list(d.columns))\n        for i, kol in enumerate(d.columns, start=1):\n            fig.add_trace(go.Scatter(x=d.index, y=d[kol], name=kol, showlegend=False), row=i, col=1)\n        fig.update_layout(height=650, template="plotly_white", margin=dict(t=40, b=10))\n        st.plotly_chart(fig)\n    except ValueError as e:\n        st.info(str(e))\n',
    'halaman/evaluasi.py': 'import streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.data_preprocessing import label_periode\nfrom modules.evaluation import PENJELASAN\nfrom modules.visualization import grafik_akurasi\n\nh = kp.perlu_hasil()\npeng = h.pengaturan\n\nst.title("5. Evaluasi Model")\nif h.akurasi is None:\n    st.info("Tidak ada periode uji, jadi akurasi ramalan tidak bisa diukur. Aktifkan "\n            "\'Sisihkan periode uji\' di halaman Konfigurasi.")\n    st.stop()\n\nst.caption(f"Akurasi **di luar sampel**: model diestimasi dengan data {label_periode(peng.awal_latih)}–"\n           f"{label_periode(peng.akhir_latih)}, lalu meramal {label_periode(peng.akhir_latih + 1)}–"\n           f"{label_periode(peng.akhir_uji)} tanpa melihat nilai aktualnya.")\n\nutama = h.akurasi.iloc[0]\nif h.pakai_x:\n    st.markdown(f"Angka di bawah: **{h.akurasi.index[0]}**. Perbandingan lengkap dengan ex-ante dan tanpa X ada "\n                "di tabel dan grafik berikutnya.")\nm = st.columns(5)\nm[0].metric("MAE", kp.fmt_angka(utama["MAE"]))\nm[1].metric("RMSE", kp.fmt_angka(utama["RMSE"]))\nm[2].metric("MAPE", f"{kp.fmt_angka(utama[\'MAPE (%)\'])}%")\nm[3].metric("sMAPE", f"{kp.fmt_angka(utama[\'sMAPE (%)\'])}%")\nm[4].metric("Theil U2", kp.fmt_angka(utama.get("Theil U2"), 3))\n\nnol = int(utama.get("Aktual = 0 (dikecualikan dari MAPE)", 0))\nif nol:\n    st.warning(f"{nol} periode uji bernilai aktual 0 dan dikecualikan dari MAPE (pembagian dengan nol). "\n               "MAE, RMSE, dan sMAPE tetap menghitungnya.")\n\nst.subheader("Perbandingan model" if h.pakai_x else "Dibandingkan dengan ramalan sederhana")\nif h.pakai_x:\n    st.caption("**ex-post (X aktual)**: X periode uji dianggap diketahui. **ex-ante (X diproyeksi SARIMA)**: X periode "\n               "uji diproyeksikan dari data latih (kondisi nyata). **tanpa X**: model yang sama tanpa variabel X. "\n               "Bila model dengan X tidak lebih akurat dari tanpa X, variabel X belum menambah daya ramal.")\nukuran = st.radio("Ukuran untuk grafik", ["MAPE (%)", "sMAPE (%)", "MAE", "RMSE", "Theil U2"], horizontal=True)\nst.plotly_chart(grafik_akurasi(h.akurasi, ukuran))\nst.caption("**Naif musiman**: nilai periode yang sama satu musim sebelumnya. **Rata-rata musiman**: rata-rata "\n           "tiap posisi musim (mis. tiap bulan) di data latih. **Naif (nilai terakhir)**: nilai terakhir data latih "\n           "diulang. Pembanding musiman hanya muncul bila s ≥ 2. Model yang baik seharusnya lebih akurat dari pembanding.")\nst.dataframe(h.akurasi.style.format(kp.fmt_angka, subset=[c for c in h.akurasi.columns\n                                                           if c not in ("n", "Aktual = 0 (dikecualikan dari MAPE)")]))\n\nst.subheader("Arti setiap ukuran")\nfor nama, isi in PENJELASAN.items():\n    st.markdown(f"- **{nama}**: {isi}")\nst.caption("Periode uji yang pendek membuat ukuran-ukuran ini sensitif terhadap satu-dua periode yang meleset jauh. "\n           "Lihat juga kolom galat per periode di halaman Hasil.")\n',
    'halaman/hasil.py': 'import pandas as pd\nimport streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.arima_model import statistik_ringkas, tabel_koefisien\nfrom modules.data_preprocessing import label_periode\nfrom modules.pipeline import tabel_hasil, tabel_perbandingan_model, tabel_perbandingan_ramalan\nfrom modules.visualization import grafik_acf, grafik_hasil\n\nh = kp.perlu_hasil()\npeng = h.pengaturan\ntarget = st.session_state.get(kp.TARGET, "Nilai")\n\nst.title("4. Hasil Proyeksi")\nfor p in h.diagnostik_akhir["peringatan"]:\n    st.warning(p)\nfor p in h.peringatan:\n    st.info(p)\n\nm = st.columns(4)\nm[0].metric("Model", h.spek.label())\nm[1].metric("Data latih", f"{label_periode(peng.awal_latih)} – {label_periode(peng.akhir_latih)}")\nm[2].metric("Periode uji", f"{label_periode(peng.akhir_latih + 1)} – {label_periode(peng.akhir_uji)}"\n            if peng.akhir_uji else "Tidak ada")\nm[3].metric("Horizon proyeksi", f"{peng.horizon} periode")\nif h.pakai_x:\n    st.info(f"**Variabel X:** {\', \'.join(h.spek.exog)}. Garis merah utama = **{h.nama_ramalan_utama}**; garis titik-titik "\n            "berwarna lain = varian lain (ex-ante / tanpa X). Klik nama di legenda untuk menyembunyikan/menampilkan garis.")\n\nst.plotly_chart(grafik_hasil(h, target))\n\nif h.pakai_x:\n    st.subheader("Perbandingan ramalan: dengan X (ex-post, ex-ante) dan tanpa X")\n    st.caption("Periode uji: model dilatih hanya dengan data latih. Ex-post memakai X aktual; ex-ante memakai X hasil "\n               "proyeksi SARIMA. Proyeksi ke depan: ex-post memakai X yang Anda isi manual di file (bila ada).")\n    tp = tabel_perbandingan_ramalan(h)\n    st.dataframe(tp.style.format({c: kp.fmt_angka for c in tp.columns if c != "Jenis"}, na_rep="—"))\n    if h.info_x:\n        with st.expander("Keterangan variabel X dan proyeksinya"):\n            st.dataframe(pd.Series(h.info_x, name="Keterangan").to_frame())\n\nst.subheader("Tabel aktual dan ramalan")\nst.caption("Baris \'Uji (backtest)\': ramalan dari model yang hanya memakai data latih. "\n           "Baris \'Proyeksi ke depan\': aktual belum ada. Semua nilai dalam skala asli data.")\ntabel = tabel_hasil(h)\nst.dataframe(tabel.style.format({c: kp.fmt_angka for c in tabel.columns if c != "Jenis"}, na_rep="—"))\n\nst.subheader("Estimasi model")\nnama_tab = ["Model untuk proyeksi", "Model backtest", "Diagnostik residual", "Ringkasan statsmodels"]\nif h.pakai_x:\n    nama_tab.insert(2, "Dengan vs tanpa X")\ntabs = st.tabs(nama_tab)\ntab_akhir, tab_bt = tabs[0], tabs[1]\ntab_diag, tab_ringkas = tabs[-2], tabs[-1]\n\nwith tab_akhir:\n    st.caption(f"Diestimasi dengan data {label_periode(peng.awal_latih)} s.d. {label_periode(h.akhir_data_proyeksi)}.")\n    st.dataframe(h.koefisien_akhir.style.format("{:.6f}"))\n    st.dataframe(h.statistik_akhir.to_frame("Nilai").style.format("{:.6f}"))\n    st.caption("Exact maximum likelihood; kriteria informasi per observasi. Bila ada differencing, "\n               "statistik dihitung pada data setelah differencing.")\n\nwith tab_bt:\n    if h.backtest is None:\n        st.info("Tidak ada periode uji.")\n    else:\n        st.caption(f"Diestimasi hanya dengan data latih {label_periode(peng.awal_latih)} s.d. "\n                   f"{label_periode(peng.akhir_latih)}; dipakai untuk ramalan periode uji.")\n        st.dataframe(tabel_koefisien(h.backtest["hasil"]).style.format("{:.6f}"))\n        st.dataframe(statistik_ringkas(h.backtest["hasil"]).to_frame("Nilai").style.format("{:.6f}"))\n\nif h.pakai_x:\n    with tabs[2]:\n        st.caption("Model tanpa X memakai orde ARMA, konstanta, dan sampel yang sama. AIC/SC lebih kecil = lebih baik; "\n                   "Prob. koefisien X < 0,05 berarti X berpengaruh signifikan.")\n        if h.backtest is not None:\n            st.markdown("**Model backtest (data latih)**")\n            st.dataframe(tabel_perbandingan_model(h, pakai_backtest=True).style.format("{:.6f}"))\n        st.markdown("**Model untuk proyeksi (seluruh data)**")\n        st.dataframe(tabel_perbandingan_model(h, pakai_backtest=False).style.format("{:.6f}"))\n        st.markdown(f"**Koefisien model tanpa X** — {h.hasil_tanpa_x.spek.label()}")\n        st.dataframe(tabel_koefisien(h.hasil_tanpa_x).style.format("{:.6f}"))\n\nwith tab_diag:\n    diag = h.diagnostik_akhir\n    c1, c2 = st.columns(2)\n    with c1:\n        st.markdown("**Ljung-Box**: p > 0,05 berarti tidak ada autokorelasi residual yang tersisa.")\n        st.dataframe(diag["ljung_box"].rename(columns={"lb_stat": "Statistik Q", "lb_pvalue": "p-value"}))\n        jb = diag["jarque_bera"]\n        st.markdown(f"**Jarque-Bera**: {jb[\'statistik\']:.3f}, p = {jb[\'p_value\']:.4f} "\n                    f"→ residual {\'normal\' if jb[\'p_value\'] > 0.05 else \'tidak normal\'}.")\n        ar, ma = diag["akar_ar_invers_maks"], diag["akar_ma_invers_maks"]\n        st.markdown(f"**Akar invers maksimum** (harus < 1): AR = {ar:.4f}" if ar is not None else "AR: –")\n        if ma is not None:\n            st.markdown(f"MA = {ma:.4f}")\n    with c2:\n        st.plotly_chart(grafik_acf(diag))\n\nwith tab_ringkas:\n    st.code(str(h.hasil_akhir.summary()), language="text")\n',
    'halaman/komponen.py': '"""Komponen tampilan bersama dan pengelolaan session state."""\nfrom __future__ import annotations\n\nimport streamlit as st\n\nfrom modules.data_preprocessing import label_periode\n\n# Kunci session state\nDF = "df_mentah"\nNAMA_FILE = "nama_file"\nSERI = "seri"\nFREKUENSI = "frekuensi"\nTARGET = "kolom_target"\nHASIL = "hasil"\nX_DASAR = "x_dasar"\nKOLOM_X = "kolom_x"\nTABEL_ORDE = "tabel_orde"\n\n\nNAVY, BIRU, ABU = "#0B1F3A", "#1D4ED8", "#F1F5F9"\n\n\ndef gaya():\n    """Tema navy - biru - putih - abu."""\n    st.markdown(\n        f"""\n        <style>\n        .block-container {{padding-top: 2rem; max-width: 1300px;}}\n        h1, h2, h3 {{color: {NAVY};}}\n        h1 {{border-bottom: 3px solid {BIRU}; padding-bottom: .35rem;}}\n        section[data-testid="stSidebar"] {{background: {NAVY};}}\n        section[data-testid="stSidebar"] * {{color: #E2E8F0 !important;}}\n        section[data-testid="stSidebar"] a[aria-current="page"],\n        section[data-testid="stSidebar"] a:hover {{background: rgba(29,78,216,.45) !important; border-radius: 8px;}}\n        div[data-testid="stMetric"] {{background: {ABU}; border-left: 4px solid {BIRU}; border-radius: 8px;\n                                       padding: 10px 14px;}}\n        div[data-testid="stMetricValue"] {{font-size: 1.4rem; color: {NAVY};}}\n        div[data-testid="stMetricLabel"] p {{color: #475569;}}\n        .stTabs [aria-selected="true"] {{color: {BIRU} !important;}}\n        .langkah {{background: {ABU}; border-top: 4px solid {BIRU}; border-radius: 10px; padding: 14px 16px;\n                   height: 100%;}}\n        .langkah b {{color: {NAVY};}}\n        </style>\n        """,\n        unsafe_allow_html=True,\n    )\n\n\ndef reset_hasil():\n    st.session_state.pop(HASIL, None)\n\n\ndef reset_data():\n    for k in (SERI, HASIL, X_DASAR, KOLOM_X, TABEL_ORDE):\n        st.session_state.pop(k, None)\n\n\ndef status_sidebar():\n    with st.sidebar:\n        st.markdown("### Status proyek")\n        nama = st.session_state.get(NAMA_FILE)\n        seri = st.session_state.get(SERI)\n        hasil = st.session_state.get(HASIL)\n        st.markdown(f"{\'✅\' if nama else \'⬜\'} Data: {nama or \'belum diunggah\'}")\n        if seri is not None:\n            terisi = seri.dropna().index\n            st.caption(f"{st.session_state.get(TARGET)} · {st.session_state.get(FREKUENSI)} · "\n                       f"{label_periode(terisi.min())} – {label_periode(terisi.max())}")\n            kolom_x = st.session_state.get(KOLOM_X) or []\n            st.caption(f"Variabel X: {\', \'.join(kolom_x) if kolom_x else \'—\'}")\n        st.markdown(f"{\'✅\' if hasil else \'⬜\'} Model: {hasil.spek.label() if hasil else \'belum dijalankan\'}")\n\n\ndef perlu_seri():\n    """Hentikan halaman bila data belum siap."""\n    if st.session_state.get(SERI) is None:\n        st.info("Belum ada data yang siap dipakai. Mulai dari halaman **Unggah Data**.")\n        st.page_link("halaman/unggah.py", label="Ke halaman Unggah Data", icon=":material/upload_file:")\n        st.stop()\n    return st.session_state[SERI]\n\n\ndef perlu_hasil():\n    if st.session_state.get(HASIL) is None:\n        st.info("Model belum dijalankan. Atur model lalu klik **Jalankan Proyeksi** di halaman Konfigurasi.")\n        st.page_link("halaman/konfigurasi.py", label="Ke halaman Konfigurasi", icon=":material/tune:")\n        st.stop()\n    return st.session_state[HASIL]\n\n\ndef fmt_angka(v, desimal: int = 2) -> str:\n    """Format angka gaya Indonesia: 1.234.567,89"""\n    try:\n        teks = f"{float(v):,.{desimal}f}"\n    except (TypeError, ValueError):\n        return str(v)\n    return teks.replace(",", "_").replace(".", ",").replace("_", ".")\n',
    'halaman/konfigurasi.py': 'import traceback\n\nimport streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules import arima_model as am\nfrom modules.data_preprocessing import (FREKUENSI, TRANSFORMASI, cek_transformasi, ke_skala_model, label_periode,\n                                        petunjuk_log_ganda)\nfrom modules.pipeline import GalatPengaturan, Pengaturan, jalankan, panjang_uji_bawaan, siapkan_desain_x\n\nseri = kp.perlu_seri()\nX_dasar = st.session_state.get(kp.X_DASAR)\nkolom_x = st.session_state.get(kp.KOLOM_X) or []\ns_bawaan = FREKUENSI[st.session_state[kp.FREKUENSI]][1]\nmusiman = s_bawaan > 1\nperiode = list(seri.dropna().index)\nn = len(periode)\n\n# nilai awal widget orde (bisa diisi otomatis oleh tombol "Cari orde terbaik")\nfor kunci, nilai in {"orde_p": 0 if musiman else 1, "orde_q": 0, "orde_P": 1 if musiman else 0,\n                     "orde_Q": 0}.items():\n    st.session_state.setdefault(kunci, nilai)\n\nst.title("3. Konfigurasi & Jalankan")\n\n# ---------------------------------------------------------------- transformasi\nst.subheader("Transformasi data")\ntransformasi = st.radio(\n    "Bagaimana data diolah sebelum dimodelkan?", list(TRANSFORMASI), format_func=TRANSFORMASI.get,\n    help="Log membuat pola musiman yang membesar-mengecil menjadi lebih stabil. Hasil ramalan selalu "\n         "dikembalikan ke skala asli (dengan exp) secara otomatis.")\nif (petunjuk := petunjuk_log_ganda(seri, transformasi)):\n    st.warning(petunjuk)\nfor g in cek_transformasi(seri, transformasi):\n    st.error(g)\nif transformasi == "sudah_log":\n    st.info("Data dianggap sudah berupa log. Model memakai data apa adanya; hasil ditampilkan dalam skala asli (exp).")\n\n# ---------------------------------------------------------------- periode\nst.subheader("Periode latih dan uji")\nst.caption("Data dibagi menurut urutan waktu (tidak diacak). Model hanya belajar dari data latih; "\n           "data uji dipakai untuk mengukur akurasi ramalan.")\npakai_uji = st.checkbox("Sisihkan periode uji (disarankan)", value=n > max(2 * s_bawaan, 10) + 10)\npanjang_uji = panjang_uji_bawaan(n, s_bawaan)\nc1, c2, c3 = st.columns(3)\nawal_latih = c1.selectbox("Awal data latih", periode, index=0, format_func=label_periode)\nopsi_akhir = [p for p in periode if p > awal_latih]\nidx_akhir = max(0, len(opsi_akhir) - 1 - (panjang_uji if pakai_uji else 0))\nakhir_latih = c2.selectbox("Akhir data latih", opsi_akhir, index=idx_akhir, format_func=label_periode)\nakhir_uji = None\nif pakai_uji:\n    opsi_uji = [p for p in periode if p > akhir_latih]\n    if opsi_uji:\n        akhir_uji = c3.selectbox("Akhir data uji", opsi_uji, index=len(opsi_uji) - 1, format_func=label_periode)\n    else:\n        c3.warning("Tidak ada data setelah akhir data latih.")\n\n# ---------------------------------------------------------------- variabel X\ntransformasi_x, lag_x, proyeksi_x = "tanpa", (0,), True\nif kolom_x:\n    st.subheader("Variabel independen (X)")\n    st.caption(f"Dipilih di halaman Unggah Data: **{\', \'.join(kolom_x)}**. Model dengan X selalu dibandingkan "\n               "dengan model yang sama tanpa X, pada sampel yang sama.")\n    c1, c2, c3 = st.columns(3)\n    transformasi_x = c1.radio("Transformasi X", ["tanpa", "log"], horizontal=True,\n                              help="Dengan \'log\', kolom dummy 0/1 tetap tidak di-log.")\n    lag_x = tuple(sorted(c2.multiselect("Lag X", list(range(0, max(2, s_bawaan) + 1)), default=[0],\n                                        help="0 = X periode yang sama; 1 = X periode sebelumnya, dst.") or [0]))\n    proyeksi_x = c3.checkbox("Proyeksikan X dengan SARIMA (ex-ante)", value=True,\n                             help="Orde SARIMA tiap X dipilih otomatis (AIC terkecil). Nilai X yang diisi manual "\n                                  "di file tetap dipakai untuk ramalan ex-post.")\n    with st.expander("Arti ex-post dan ex-ante"):\n        st.markdown("- **Ex-post**: ramalan memakai X yang sudah diketahui (X aktual di periode uji, atau X yang "\n                    "Anda isi manual untuk periode ke depan).\\n"\n                    "- **Ex-ante**: ramalan memakai X hasil proyeksi SARIMA, yaitu kondisi nyata saat X belum diketahui.\\n"\n                    "- **Tanpa X**: model yang sama tanpa variabel X, sebagai pembanding.")\n\n# ---------------------------------------------------------------- model\nst.subheader("Spesifikasi model")\nwith st.expander("Penjelasan parameter"):\n    st.markdown(\n        """\n| Parameter | Arti |\n|---|---|\n| **p** | Jumlah lag AR: pengaruh nilai periode-periode sebelumnya |\n| **d** | Berapa kali data dibedakan (differencing). 0 bila data sudah stasioner |\n| **q** | Jumlah lag MA: pengaruh kesalahan ramalan sebelumnya |\n| **P, D, Q** | Sama seperti p, d, q tetapi pada lag musiman (mis. 12 bulan lalu) |\n| **s** | Panjang satu musim: 12 bulanan, 4 triwulanan, 52 mingguan, 7 harian, 1 = tanpa musim |\n| **Konstanta C** | Rata-rata deret; bila ada differencing (d/D > 0), C menjadi drift |\n"""\n    )\nc = st.columns(7)\np = c[0].number_input("p", 0, 5, key="orde_p")\nd = c[1].number_input("d", 0, 2, 0)\nq = c[2].number_input("q", 0, 5, key="orde_q")\nP = c[3].number_input("P", 0, 3, key="orde_P")\nD = c[4].number_input("D", 0, 1, 0)\nQ = c[5].number_input("Q", 0, 3, key="orde_Q")\ns = c[6].number_input("s", 1, 366, s_bawaan)\nkonstanta = st.checkbox("Sertakan konstanta C (rata-rata)", value=True)\nif konstanta and (d or D):\n    st.warning("Dengan differencing (d atau D > 0), konstanta berarti tren/drift; X juga ikut di-differencing.")\nif s < 2 and (P or D or Q):\n    st.error("Untuk data tanpa musim (s = 1), isi P = D = Q = 0.")\n\nwith st.expander("Opsi lanjutan"):\n    paksa_stasioner = st.checkbox("Paksa model stasioner", value=False,\n                                  help="Bawaan: tidak dipaksa (exact maximum likelihood tanpa batasan).")\n    paksa_invertible = st.checkbox("Paksa model invertible", value=False)\n    koreksi_bias = st.checkbox("Koreksi bias saat mengembalikan dari log", value=False,\n                               help="exp(ramalan log) adalah median, bukan rata-rata. Koreksi mengalikan "\n                                    "dengan exp(σ²/2). Bawaan: tidak dipakai.")\n\n# ---------------------------------------------------------------- proyeksi\nst.subheader("Proyeksi")\nc1, c2 = st.columns(2)\nhorizon = c1.number_input("Horizon (jumlah periode ke depan)", 1, 5 * s_bawaan if musiman else 60,\n                          s_bawaan if musiman else 5)\ntingkat = c2.select_slider("Tingkat kepercayaan selang ramalan", [80, 90, 95, 99], value=95)\n\nspek = am.SpesifikasiModel(order=(int(p), int(d), int(q)), seasonal_order=(int(P), int(D), int(Q), int(s)),\n                           konstanta=konstanta, paksa_stasioner=paksa_stasioner, paksa_invertible=paksa_invertible)\npeng = Pengaturan(spek=spek, transformasi=transformasi, awal_latih=awal_latih, akhir_latih=akhir_latih,\n                  akhir_uji=akhir_uji, horizon=int(horizon), tingkat_kepercayaan=int(tingkat),\n                  koreksi_bias=koreksi_bias, kolom_x=tuple(kolom_x), transformasi_x=transformasi_x, lag_x=lag_x,\n                  proyeksi_x_sarima=proyeksi_x)\n\n\n# ---------------------------------------------------------------- pemilihan orde otomatis\ndef cari_orde(maks, kriteria):\n    """Callback: estimasi semua kombinasi ARMA pada data latih, isi p, q, P, Q dengan yang terbaik."""\n    try:\n        _, X_desain, galat = siapkan_desain_x(X_dasar, peng)\n        if galat:\n            st.session_state[kp.TABEL_ORDE] = ("galat", galat)\n            return\n        y = ke_skala_model(seri, transformasi).loc[awal_latih:akhir_latih]\n        tabel, daftar = am.pilih_orde(y, int(s), X=X_desain,\n                                      exog=tuple(X_desain.columns) if X_desain is not None else (),\n                                      konstanta=konstanta, d=int(d), D=int(D), maks_p=maks[0], maks_q=maks[1],\n                                      maks_P=maks[2], maks_Q=maks[3], kriteria=kriteria)\n        terbaik = daftar[0]\n        st.session_state["orde_p"], st.session_state["orde_q"] = terbaik.order[0], terbaik.order[2]\n        st.session_state["orde_P"], st.session_state["orde_Q"] = terbaik.seasonal_order[0], terbaik.seasonal_order[2]\n        st.session_state[kp.TABEL_ORDE] = ("ok", tabel, kriteria)\n        kp.reset_hasil()\n    except Exception as e:  # tampilkan, jangan hentikan halaman\n        st.session_state[kp.TABEL_ORDE] = ("galat", [f"Pencarian orde gagal: {e}"])\n\n\nwith st.expander("Pilih orde otomatis (p, q, P, Q)", expanded=kp.TABEL_ORDE in st.session_state):\n    st.caption("Mengestimasi semua kombinasi ARMA hingga batas maksimum pada **data latih** (d, D, s, konstanta, dan X "\n               "mengikuti pengaturan di atas), lalu mengisi p, q, P, Q dengan model yang kriterianya terkecil.")\n    c = st.columns(5)\n    maks = (c[0].number_input("Maks p", 0, 4, 2), c[1].number_input("Maks q", 0, 4, 2),\n            c[2].number_input("Maks P", 0, 2, 1 if s >= 2 else 0, disabled=s < 2),\n            c[3].number_input("Maks Q", 0, 2, 1 if s >= 2 else 0, disabled=s < 2))\n    kriteria = c[4].selectbox("Kriteria", ["AIC", "SC", "HQ"])\n    jumlah = (maks[0] + 1) * (maks[1] + 1) * ((maks[2] + 1) * (maks[3] + 1) if s >= 2 else 1)\n    st.button(f"Cari orde terbaik ({jumlah} model)", icon=":material/search:", on_click=cari_orde,\n              args=(tuple(int(v) for v in maks), kriteria))\n    hasil_orde = st.session_state.get(kp.TABEL_ORDE)\n    if hasil_orde and hasil_orde[0] == "galat":\n        for g in hasil_orde[1]:\n            st.error(g)\n    elif hasil_orde:\n        _, tabel, krit = hasil_orde\n        st.success(f"Terpilih ARMA {tabel.iloc[0][\'ARMA (p,q)(P,Q)\']} · {krit} = {tabel.iloc[0][krit]:.6f} · "\n                   f"{len(tabel)} model diestimasi, {int((~tabel[\'Konvergen\']).sum())} tidak konvergen, "\n                   f"{int(tabel[\'Batasan stasioner\'].sum())} dengan batasan stasioner. Isian p, q, P, Q sudah diperbarui.")\n        st.dataframe(tabel.head(10).style.format({"AIC": "{:.6f}", "SC": "{:.6f}", "HQ": "{:.6f}",\n                                                  "Log likelihood": "{:.4f}"}))\n\nst.divider()\nlabel_x = f" + X ({\', \'.join(kolom_x)}; lag {\', \'.join(map(str, lag_x))})" if kolom_x else ""\nst.markdown(f"**Model:** {spek.label()}{label_x} · **Latih:** {label_periode(awal_latih)}–{label_periode(akhir_latih)}"\n            + (f" · **Uji:** {label_periode(akhir_latih + 1)}–{label_periode(akhir_uji)}" if akhir_uji else " · tanpa uji")\n            + f" · **Horizon:** {horizon}")\n\nif st.button("Jalankan Proyeksi", type="primary", icon=":material/play_arrow:"):\n    with st.status("Menjalankan model...", expanded=True) as status:\n        try:\n            st.write("Memeriksa data dan pengaturan, mengestimasi model"\n                     + (", memproyeksikan X, dan membandingkan dengan model tanpa X" if kolom_x else "")\n                     + ", membuat ramalan...")\n            hasil = jalankan(seri, peng, X_dasar)\n            st.session_state[kp.HASIL] = hasil\n            status.update(label="Selesai", state="complete")\n        except GalatPengaturan as e:\n            status.update(label="Pengaturan perlu diperbaiki", state="error")\n            for baris in str(e).splitlines():\n                st.error(baris)\n        except Exception as e:\n            status.update(label="Estimasi gagal", state="error")\n            st.error("Model tidak berhasil diestimasi dengan pengaturan ini. Coba model yang lebih sederhana, "\n                     "periksa nilai kosong, atau aktifkan opsi \'Paksa model stasioner\'.")\n            with st.expander("Detail teknis (untuk pengembang)"):\n                st.code(f"{type(e).__name__}: {e}\\n\\n{traceback.format_exc()}")\n\nif st.session_state.get(kp.HASIL) is not None:\n    st.success(f"Hasil tersedia untuk {st.session_state[kp.HASIL].spek.label()}.")\n    if st.button("Lihat hasil", icon=":material/show_chart:"):\n        st.switch_page("halaman/hasil.py")\n',
    'halaman/unduh.py': 'import re\n\nimport streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules import export_results as ex\nfrom modules.visualization import png_hasil\n\nh = kp.perlu_hasil()\ntarget = st.session_state.get(kp.TARGET, "Nilai")\ndasar = re.sub(r"[^A-Za-z0-9_-]+", "_", f"proyeksi_{target}").strip("_")\n\nst.title("6. Unduh Hasil")\nst.caption(f"Semua file dibuat dari model yang terakhir dijalankan: **{h.spek.label()}**.")\n\nc1, c2 = st.columns(2)\nwith c1:\n    st.markdown("**Laporan lengkap (Excel)**  \\nPengaturan, hasil, evaluasi, koefisien, statistik model.")\n    st.download_button("Unduh laporan .xlsx", ex.excel_laporan(h), f"{dasar}_laporan.xlsx",\n                       "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",\n                       type="primary", icon=":material/table_view:")\n    st.markdown("**Hasil ramalan (Excel)**")\n    st.download_button("Unduh hasil .xlsx", ex.excel_hasil_saja(h), f"{dasar}.xlsx",\n                       "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", icon=":material/grid_on:")\n    st.markdown("**Hasil ramalan (CSV)**")\n    st.download_button("Unduh hasil .csv", ex.csv_hasil(h), f"{dasar}.csv", "text/csv", icon=":material/description:")\nwith c2:\n    st.markdown("**Grafik (PNG)**")\n    png = png_hasil(h, target)\n    st.image(png)\n    st.download_button("Unduh grafik .png", png, f"{dasar}.png", "image/png", icon=":material/image:")\n    st.markdown("**Ringkasan model (TXT)**")\n    st.download_button("Unduh ringkasan .txt", ex.teks_ringkasan(h), f"{dasar}_ringkasan.txt", "text/plain",\n                       icon=":material/article:")\n',
    'halaman/unggah.py': 'from pathlib import Path\n\nimport pandas as pd\nimport streamlit as st\n\nfrom halaman import komponen as kp\nfrom modules.data_loader import GalatData, baca_file, ringkasan_dataset, tebak_kolom\nfrom modules.data_preprocessing import (FREKUENSI, adalah_dummy, kolom_x_otomatis, label_periode, siapkan_seri,\n                                        siapkan_x, tebak_frekuensi)\n\nCONTOH = Path(__file__).resolve().parent.parent / "sample_data" / "sample_dataset.xlsx"\n\nst.title("1. Unggah Data")\nst.caption("Unggah file Excel (.xlsx) atau CSV (.csv). Satu baris = satu periode. Kolom angka lain selain target "\n           "bisa dipakai sebagai variabel independen (X) untuk SARIMAX; baris tanggal ke depan dengan target kosong "\n           "dibaca sebagai nilai X masa depan.")\n\nc1, c2 = st.columns([3, 1])\nunggahan = c1.file_uploader("Pilih file", type=["xlsx", "xls", "csv"])\nc2.write("")\nc2.write("")\npakai_contoh = c2.button("Pakai data contoh", help="Data sintetis untuk mencoba aplikasi.")\n\nid_unggahan = getattr(unggahan, "file_id", None) or (unggahan.name if unggahan else None)\nif unggahan is not None and id_unggahan != st.session_state.get("id_unggahan"):\n    st.session_state["id_unggahan"] = id_unggahan  # hanya dibaca sekali per file baru\n    try:\n        st.session_state[kp.DF] = baca_file(unggahan)\n        st.session_state[kp.NAMA_FILE] = unggahan.name\n        kp.reset_data()\n    except GalatData as e:\n        st.error(str(e))\n        st.stop()\nelif pakai_contoh:\n    st.session_state[kp.DF] = baca_file(CONTOH)\n    st.session_state[kp.NAMA_FILE] = "sample_dataset.xlsx (contoh sintetis)"\n    kp.reset_data()\n\ndf = st.session_state.get(kp.DF)\nif df is None:\n    st.info("Belum ada file. Unggah file Anda atau klik **Pakai data contoh**.")\n    st.stop()\n\nst.success(f"File: **{st.session_state[kp.NAMA_FILE]}**")\nringkas = ringkasan_dataset(df)\nm1, m2, m3 = st.columns(3)\nm1.metric("Jumlah baris (observasi)", ringkas["jumlah_baris"])\nm2.metric("Jumlah kolom (variabel)", ringkas["jumlah_kolom"])\nm3.metric("Total sel kosong", int(df.isna().sum().sum()))\n\nt1, t2 = st.tabs(["Pratinjau data", "Informasi kolom"])\nt1.dataframe(df.head(50))\nt2.dataframe(ringkas["info_kolom"])\n\nst.subheader("Pilih kolom dan frekuensi")\ntebak_tgl, tebak_target = tebak_kolom(df)\nkolom = list(df.columns)\nc1, c2, c3 = st.columns(3)\nkolom_tgl = c1.selectbox("Kolom tanggal/periode", kolom, index=kolom.index(tebak_tgl),\n                         help="Format yang dikenali: 2021-01-31, 31/01/2021, tanggal Excel, 2021M01, 2021Q1, 2021.")\nkolom_target = c2.selectbox("Variabel yang diramal (target)", kolom, index=kolom.index(tebak_target),\n                            help="Kolom angka yang ingin diproyeksikan, dalam satuan aslinya.")\nfrek_tebakan = tebak_frekuensi(df[kolom_tgl])\nfrekuensi = c3.selectbox("Frekuensi data", list(FREKUENSI), index=list(FREKUENSI).index(frek_tebakan),\n                         help="Ditebak otomatis dari kolom tanggal; ubah bila perlu. Menentukan panjang musim bawaan.")\n\nvalidasi = siapkan_seri(df, kolom_tgl, kolom_target, frekuensi)\nfor p in validasi.peringatan:\n    st.warning(p)\nif not validasi.ok:\n    for g in validasi.galat:\n        st.error(g)\n    st.session_state.pop(kp.SERI, None)\n    st.stop()\n\nseri = validasi.seri\nakhir_y = seri.last_valid_index()\n\n# ---------------------------------------------------------------- variabel X\nst.subheader("Variabel independen (X) — opsional")\nkandidat_x = [k for k in kolom if k not in (kolom_tgl, kolom_target)]\nbawaan_x = [k for k in kolom_x_otomatis(df, kolom_tgl, kolom_target)]\nkolom_x = st.multiselect(\n    "Kolom X untuk SARIMAX", kandidat_x, default=bawaan_x,\n    help="Bawaan: semua kolom angka selain tanggal dan target. Kosongkan untuk SARIMA biasa. "\n         "Keluarkan kolom yang bukan variabel penjelas (mis. kolom hasil ramalan software lain).")\nX_dasar = None\nif kolom_x:\n    vx = siapkan_x(df, kolom_tgl, kolom_x, frekuensi, seri.index)\n    for p in vx.peringatan:\n        st.warning(p)\n    if not vx.ok:\n        for g in vx.galat:\n            st.error(g)\n        st.stop()\n    X_dasar = vx.X\n    ringkas_x = pd.DataFrame({\n        "Jenis": ["dummy (0/1)" if adalah_dummy(X_dasar[k]) else "kontinu" for k in kolom_x],\n        "Terisi s.d. akhir target": [int(X_dasar[k].loc[:akhir_y].notna().sum()) for k in kolom_x],\n        "Kosong s.d. akhir target": [int(X_dasar[k].loc[:akhir_y].isna().sum()) for k in kolom_x],\n        "Nilai manual ke depan": [int(X_dasar[k].loc[akhir_y + 1:].notna().sum()) for k in kolom_x],\n    }, index=pd.Index(kolom_x, name="Variabel X"))\n    st.dataframe(ringkas_x)\n    if ringkas_x["Kosong s.d. akhir target"].any():\n        st.warning("Ada nilai X yang kosong di periode historis. Model membutuhkan X lengkap di periode estimasi; "\n                   "lengkapi data atau keluarkan kolom tersebut.")\n    if not ringkas_x["Nilai manual ke depan"].any():\n        st.info("Belum ada nilai X untuk periode ke depan. Proyeksi akan memakai X hasil proyeksi SARIMA (ex-ante); "\n                "untuk ex-post, tambahkan baris tanggal ke depan berisi nilai X (target dikosongkan).")\nelse:\n    st.caption("Tanpa X: model SARIMA biasa.")\n\nlama = st.session_state.get(kp.SERI)\nlama_x = st.session_state.get(kp.X_DASAR)\nif (lama is None or not lama.equals(seri) or st.session_state.get(kp.TARGET) != kolom_target\n        or st.session_state.get(kp.KOLOM_X) != kolom_x\n        or (lama_x is None) != (X_dasar is None) or (X_dasar is not None and not X_dasar.equals(lama_x))):\n    kp.reset_hasil()\n    st.session_state.pop(kp.TABEL_ORDE, None)\nst.session_state[kp.SERI] = seri\nst.session_state[kp.FREKUENSI] = frekuensi\nst.session_state[kp.TARGET] = kolom_target\nst.session_state[kp.KOLOM_X] = kolom_x\nst.session_state[kp.X_DASAR] = X_dasar\n\nterisi = seri.loc[:akhir_y]\nm1, m2, m3, m4 = st.columns(4)\nm1.metric("Rentang data target", f"{label_periode(terisi.index.min())} – {label_periode(akhir_y)}")\nm2.metric("Jumlah periode", len(terisi))\nm3.metric("Periode kosong", int(terisi.isna().sum()))\nm4.metric("Variabel X", len(kolom_x))\nst.success("Data siap. Lanjutkan ke eksplorasi atau langsung ke konfigurasi model.")\nc1, c2 = st.columns(2)\nif c1.button("Eksplorasi data", icon=":material/query_stats:"):\n    st.switch_page("halaman/eksplorasi.py")\nif c2.button("Konfigurasi model", type="primary", icon=":material/tune:"):\n    st.switch_page("halaman/konfigurasi.py")\n',
    '.streamlit/config.toml': '[theme]\nbase = "light"\nprimaryColor = "#1D4ED8"              # biru\nbackgroundColor = "#FFFFFF"           # putih\nsecondaryBackgroundColor = "#F1F5F9"  # abu muda\ntextColor = "#0B1F3A"                 # navy\nfont = "sans serif"\n\n[server]\nmaxUploadSize = 50\n\n[browser]\ngatherUsageStats = false\n',
    'requirements.txt': 'streamlit>=1.40\npandas>=2.1\nnumpy>=1.26\nscipy>=1.11\nstatsmodels>=0.14.2\nplotly>=5.20\nmatplotlib>=3.8\nopenpyxl>=3.1\n',
}
for _nama, _isi in BERKAS_WEB.items():
    (FOLDER / _nama).parent.mkdir(parents=True, exist_ok=True)
    (FOLDER / _nama).write_text(_isi)

!pip -q install streamlit
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
!cd /content/sarima_app && nohup python -m streamlit run app.py --server.port 8501 --server.headless true &> /content/streamlit.log &
!nohup /content/cloudflared tunnel --url http://localhost:8501 &> /content/cloudflared.log &
import time, re as _re
for _ in range(30):
    time.sleep(2)
    _m = _re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if _m:
        print("🌐 Buka aplikasi di:", _m.group(0))
        break
else:
    print("Tautan belum muncul. Tunggu sebentar lalu jalankan ulang sel ini, atau lihat /content/cloudflared.log")